# 04 — NIAH retention curves and the control battery

**Stage:** Proposal Stages 3–4 on a single checkpoint. **Produces:** Table 4 (controls
C1–C4), Table 6 (retention summary), and the raw rows behind Figures 4–7.

### What changed from the pilot

| pilot | here | why |
|---|---|---|
| decoded `ahn_raw` (pre-`o_proj`) | decodes `o_t = o_proj(ahn_raw)` | the pilot's vector was in head-concat space; dims coincide at 3B so it ran and returned noise |
| `vec @ unembed.T` | `readout_logits` with final RMSNorm | Qwen applies `model.norm` before `lm_head` |
| C1 as `o_t(AHN) − o_t(NOWRITE)` ≡ `o_t` | C1 on the **residual stream** | the AHN output is zero under NOWRITE by construction, so the control was vacuous |
| needle at token ~5 | needle placed past `num_attn_sinks` | tokens in the sink prefix are never compressed |
| best layer chosen on the same data it is plotted from | layers fixed in advance | selection-on-test |
| 2 needles (a third silently dropped) | needles filtered up front | cohort size becomes a decision |
| no CIs, no fit diagnostics | bootstrap CIs and R² | Table 6's R² column decides whether "half-life" is even meaningful |

**Prerequisite:** notebook 01 gates pass and `02_table3_jlens_validation.json` says
`TABLE_3_PASSED: true`. If the lens is not validated, run this with `USE_JLENS=False`
to get logit-lens numbers and label every figure "logit lens, preliminary" — that is a
legitimate pilot, but it is not RQ2.


In [1]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks
working directory pinned to /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks


In [2]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = "/workspace/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/workspace/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [3]:
EXP = dict(
    layers              = [9, 18, 27],          # fixed in advance, matches the J-lens map
    # Prompt length is roughly num_attn_sinks + sliding_window + eviction_distance, so
    # each distance sets the cost of its own conditions: 8192 -> ~16.4K tokens, 16384 ->
    # ~24.6K. Those two dominated the sweep budget. 16384 is dropped from run 1 and added
    # back only if the decay curve has not flattened by 8192 -- six points still support
    # the exponential fit, and Table 6's R2 column is what says whether it does.
    #
    # distance=0 is also dropped: build_niah_prompt puts the needle at ~145 and the
    # compression boundary at n - sliding_window, which for distance=0 lands at ~146, so
    # the ACTUAL eviction distance is ~1 token and the needle_is_evicted check
    # (sinks <= needle_pos < window_start) is one token from failing. Some filler
    # variants would be silently dropped. 64 is the smallest distance that is safely
    # past the boundary for every filler.
    eviction_distances  = [64, 256, 512, 1024, 2048, 4096, 8192],   # add 16384 if needed
    needle_candidates   = ["Paris", "banana", "Tokyo", "violin", "cinnamon",
                           "harbour", "lantern", "sapphire", "meadow", "trumpet"],
    n_filler_variants   = 3,                    # repeats per (needle, distance)
    use_jlens           = True,
    jlens_path          = os.path.join(CFG["results_dir"], "jlens_qwen25_3b.pt"),
)
print(json.dumps(EXP, indent=2))


{
  "layers": [
    9,
    18,
    27
  ],
  "eviction_distances": [
    64,
    256,
    512,
    1024,
    2048,
    4096,
    8192
  ],
  "needle_candidates": [
    "Paris",
    "banana",
    "Tokyo",
    "violin",
    "cinnamon",
    "harbour",
    "lantern",
    "sapphire",
    "meadow",
    "trumpet"
  ],
  "n_filler_variants": 3,
  "use_jlens": true,
  "jlens_path": "results/run_3b_gdn/jlens_qwen25_3b.pt"
}


In [4]:
import torch, numpy as np, time
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok, probe = bundle.tokenizer, ai.AHNProbe(bundle)

# Loading the lens and checking Table 3 are two separate questions and used to share a
# try/except. That was a hard blocker: TABLE_3_PASSED is currently False (checks 2 and 3
# fail, see notebook 02), the `except` caught FileNotFoundError only, so the AssertionError
# escaped and killed the notebook here -- before a single measurement -- even though
# README "Next steps" item 4 explicitly says to run this WITH the J-lens.
#
# Now: a missing .pt falls back to the logit lens (unchanged behaviour), a missing or
# failing Table 3 is a loud warning that stamps lens_validated=False onto every saved row.
lens = None
LENS_VALIDATED = False

if EXP["use_jlens"]:
    try:
        lens = ai.JacobianLens.load(EXP["jlens_path"], map_location=str(bundle.model.device))
        print("J-lens loaded, layers:", sorted(lens.jacobians))
    except FileNotFoundError:
        print("! no J-lens found at", EXP["jlens_path"])
        print("  Falling back to LOGIT LENS. Label every figure 'logit lens, preliminary'.")
        print("  This is not RQ2.")
        EXP["use_jlens"] = False

if EXP["use_jlens"]:
    try:
        v = ai.load_json("02_table3_jlens_validation.json")
        LENS_VALIDATED = bool(v.get("TABLE_3_PASSED"))
        print("Table 3 passed:", LENS_VALIDATED)
    except FileNotFoundError:
        print("! 02_table3_jlens_validation.json not found in", CFG["results_dir"])
        print("  It was produced on the GPU box by notebook 02 but never downloaded.")
        print("  Proceeding with lens_validated=False.")

    if not LENS_VALIDATED:
        print()
        print("!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.")
        print("   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known")
        print("   facts. It does beat the plain logit lens by 8-204x on the same prompts,")
        print("   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.")
        print("   This notebook's control battery (C1/C2/C4) tests that property directly,")
        print("   which is exactly the evidence Gautam asked for before ruling on the lens.")
        print("   Every row is stamped lens_validated=False; label every figure")
        print("   'J-lens, not validated on Table 3' until that ruling lands.")

READOUT = "jlens" if EXP["use_jlens"] else "logit_lens"
EXP["lens_validated"] = LENS_VALIDATED
print()
print("readout:", READOUT, "| lens_validated:", LENS_VALIDATED)


/venv/ahn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.07s/it]

J-lens loaded, layers: [9, 18, 27]
Table 3 passed: False

!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.
   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known
   facts. It does beat the plain logit lens by 8-204x on the same prompts,
   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.
   This notebook's control battery (C1/C2/C4) tests that property directly,
   which is exactly the evidence Gautam asked for before ruling on the lens.
   Every row is stamped lens_validated=False; label every figure
   'J-lens, not validated on Table 3' until that ruling lands.

readout: jlens | lens_validated: False


In [5]:
needles = ai.single_token_needles(tok, EXP["needle_candidates"])
assert len(needles) >= 5, "need at least 5 single-token needles for a usable cohort"

# distractors for control C2: semantically near the needle, absent from the context
DISTRACTORS = {"Paris": "London", "banana": "mango", "Tokyo": "Osaka",
               "violin": "cello", "cinnamon": "nutmeg", "harbour": "wharf",
               "lantern": "torch", "sapphire": "emerald", "meadow": "pasture",
               "trumpet": "clarinet"}
distractor_ids = {}
for n in needles:
    d = DISTRACTORS.get(n)
    ids = tok.encode(f" {d}", add_special_tokens=False) if d else []
    if len(ids) == 1:
        distractor_ids[n] = ids[0]
print(f"{len(distractor_ids)}/{len(needles)} needles have a single-token distractor")


dropped multi-token needles: {'sapphire': 2, 'meadow': 2}
kept 8 single-token needles: ['Paris', 'Tokyo', 'banana', 'cinnamon', 'harbour', 'lantern', 'trumpet', 'violin']
4/8 needles have a single-token distractor


## The measurement

One row per `(needle, eviction distance, filler variant, layer)`. Each row carries
everything Tables 4, 6 and 8 need, plus the four controls, so the whole battery comes
out of one sweep rather than four.


In [6]:
@torch.no_grad()
def measure(needle, needle_id, distance, filler_idx, in_window=False, shuffle=False):
    spec = ai.build_niah_prompt(tok, needle, bundle, eviction_distance=distance,
                                in_window=in_window, filler_idx=filler_idx)
    if not spec["ahn_will_activate"]:
        return []
    if not in_window and not spec["needle_is_evicted"]:
        return []

    prompt = spec["prompt"]
    if shuffle:   # control C3 — destroy word order, keep the token multiset
        w = prompt.split()
        np.random.default_rng(ai.SEED).shuffle(w)
        prompt = " ".join(w)

    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    on  = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=True)
    off = probe.run(ins, nowrite=True,  layers=EXP["layers"], capture_residual=True)

    out = []
    for L in EXP["layers"]:
        if L not in on.ahn_raw:
            continue
        o_t = on.o_t(L, pos=-1)
        lg  = ai.readout_logits(o_t, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        # C1: residual-stream difference, the non-vacuous zero-state control
        d_res = on.residual(L, pos=-1).float() - off.residual(L, pos=-1).float()
        lg_c1 = ai.readout_logits(d_res, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        row = {
            "needle": needle, "layer": L, "readout": READOUT,
            # travels with the data so the caveat cannot be lost between here and
            # a figure caption -- see the lens block above
            "lens_validated": LENS_VALIDATED,
            "requested_distance": distance,
            "eviction_distance": spec["actual_eviction_distance"],
            "in_window": in_window, "shuffled": shuffle, "filler_idx": filler_idx,
            "n_tokens": spec["n_tokens"], "needle_pos": spec["needle_pos"],
            "rank": ai.token_rank(lg, needle_id),
            "p_mem": ai.token_prob(lg, needle_id),
            "entropy": ai.readout_entropy(lg),
            "o_t_norm": float(o_t.float().norm()),
            "rank_c1_residual": ai.token_rank(lg_c1, needle_id),
            "p_mem_c1_residual": ai.token_prob(lg_c1, needle_id),
        }
        if needle in distractor_ids:      # C2
            row["rank_distractor"] = ai.token_rank(lg, distractor_ids[needle])
            row["p_distractor"] = ai.token_prob(lg, distractor_ids[needle])
        out.append(row)
    return out


In [7]:
rows, t0 = [], time.time()
total = len(needles) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for needle, nid in needles.items():
    for dist in EXP["eviction_distances"]:
        for fi in range(EXP["n_filler_variants"]):
            rows += measure(needle, nid, dist, fi)
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{total}] {len(rows)} rows, {(time.time()-t0)/60:.1f} min")
            ai.free_cuda()
print(f"main sweep: {len(rows)} rows in {(time.time()-t0)/60:.1f} min")


[20/168] 60 rows, 1.4 min
[40/168] 120 rows, 2.4 min
[60/168] 180 rows, 3.2 min
[80/168] 240 rows, 3.7 min
[100/168] 300 rows, 4.4 min
[120/168] 360 rows, 5.5 min
[140/168] 420 rows, 6.4 min
[160/168] 480 rows, 7.1 min
main sweep: 504 rows in 7.4 min


In [8]:
# C4 pre-eviction baseline (the ceiling) and C3 shuffled context
ctrl_rows = []
for needle, nid in needles.items():
    for fi in range(EXP["n_filler_variants"]):
        ctrl_rows += measure(needle, nid, 0, fi, in_window=True)          # C4 ceiling
    for dist in (1024, 4096):
        ctrl_rows += measure(needle, nid, dist, 0, shuffle=True)          # C3
    ai.free_cuda()
print(f"control rows: {len(ctrl_rows)}")

all_rows = rows + ctrl_rows
ai.save_json({"rows": all_rows, "cfg": CFG, "exp": EXP,
              "needles": needles, "distractors": distractor_ids},
             "04_retention_rows.json")
print("saved -> 04_retention_rows.json  (this is the file notebook 05 reads)")


control rows: 120
saved -> 04_retention_rows.json  (this is the file notebook 05 reads)


## Table 4 — the control battery, evaluated

C1 and C4 have to pass before Table 6 is filled in. C3 failing is *not* a bug — the
Expected-Tables document flags it as potentially the most publishable result in the
project: if shuffling the context barely changes retention, AHN is closer to a learned
recency mechanism than to content memory, which contradicts the framing of the original
AHN paper.


In [9]:
import numpy as np
V = bundle.vocab
def sel(**kw):
    out = all_rows
    for k, v in kw.items():
        out = [r for r in out if r.get(k) == v]
    return out

main   = [r for r in all_rows if not r["in_window"] and not r["shuffled"]]
inwin  = [r for r in all_rows if r["in_window"]]
shuf   = [r for r in all_rows if r["shuffled"]]

T4 = {}

# C1 — the memory's contribution must beat what the residual difference alone explains,
#      and both must beat chance.
T4["C1_zero_state"] = {
    "mean_rank_o_t": float(np.mean([r["rank"] for r in main])),
    "mean_rank_residual_delta": float(np.mean([r["rank_c1_residual"] for r in main])),
    "chance_rank": V / 2,
    "passed": bool(np.mean([r["rank"] for r in main]) < V / 10),
    "note": "fails if the target is at chance in the memory readout: nothing is retained, "
            "or the readout is still in the wrong basis",
}

# C2 — the true needle must beat a semantically near absent token by >= 1 order of magnitude
withd = [r for r in main if "p_distractor" in r]
if withd:
    ratio = float(np.median([(r["p_mem"] + 1e-12) / (r["p_distractor"] + 1e-12) for r in withd]))
    T4["C2_distractor"] = {"median_prob_ratio": ratio, "n": len(withd),
                           "passed": bool(ratio >= 10.0),
                           "note": "below 10x: the readout reflects topic, not the stored item; "
                                   "RQ2 weakens to 'semantic gist'"}

# C3 — shuffling should hurt retention if the state stores content rather than recency
if shuf:
    T4["C3_shuffled_context"] = {
        "mean_rank_ordered": float(np.mean([r["rank"] for r in main
                                            if r["requested_distance"] in (1024, 4096)])),
        "mean_rank_shuffled": float(np.mean([r["rank"] for r in shuf])),
        "passed": bool(np.mean([r["rank"] for r in shuf])
                       > np.mean([r["rank"] for r in main
                                  if r["requested_distance"] in (1024, 4096)])),
        "note": "FAILURE HERE IS A FINDING, not a bug — see Expected_Tables_and_Figures §3",
    }

# C4 — pre-eviction ceiling must be BETTER than any evicted condition.
#      In the pilot it was worse (Paris baseline rank 110712 vs ~95000 evicted), which
#      is the single clearest sign the measurement was not measuring retention.
if inwin:
    T4["C4_pre_eviction_baseline"] = {
        "mean_rank_in_window": float(np.mean([r["rank"] for r in inwin])),
        "mean_rank_evicted": float(np.mean([r["rank"] for r in main])),
        "passed": bool(np.mean([r["rank"] for r in inwin])
                       < np.mean([r["rank"] for r in main])),
        "note": "if the in-window ceiling is worse than the evicted condition, the "
                "placement or the readout is wrong. Stop and fix before Table 6.",
    }

T4["BATTERY_PASSED"] = bool(T4["C1_zero_state"]["passed"]
                            and T4.get("C4_pre_eviction_baseline", {}).get("passed", True))
ai.save_json(T4, "04_table4_controls.json")
print(json.dumps(T4, indent=2))
print("\nC1+C4:", "PASS -> Table 6 may be populated" if T4["BATTERY_PASSED"]
      else "FAIL -> fix instrumentation, do NOT report Table 6")


{
  "C1_zero_state": {
    "mean_rank_o_t": 87687.68055555556,
    "mean_rank_residual_delta": 100491.29365079365,
    "chance_rank": 75968.0,
    "passed": false,
    "note": "fails if the target is at chance in the memory readout: nothing is retained, or the readout is still in the wrong basis"
  },
  "C2_distractor": {
    "median_prob_ratio": 1.0000000000002294,
    "n": 252,
    "passed": false,
    "note": "below 10x: the readout reflects topic, not the stored item; RQ2 weakens to 'semantic gist'"
  },
  "C3_shuffled_context": {
    "mean_rank_ordered": 88702.29861111111,
    "mean_rank_shuffled": 72769.10416666667,
    "passed": false,
    "note": "FAILURE HERE IS A FINDING, not a bug \u2014 see Expected_Tables_and_Figures \u00a73"
  },
  "C4_pre_eviction_baseline": {
    "mean_rank_in_window": 77429.51388888889,
    "mean_rank_evicted": 87687.68055555556,
    "passed": true,
    "note": "if the in-window ceiling is worse than the evicted condition, the placement or the readout 

## Adding more metrics

In [10]:
import json, numpy as np
data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]

for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    print(f"layer {L}: mean_rank={np.mean([r['rank'] for r in rs]):.0f}  "
          f"mean_p_mem={np.mean([r['p_mem'] for r in rs]):.3e}")

# raw p_mem / p_distractor pairs, unrounded, no epsilon
withd = [r for r in main if "p_distractor" in r][:10]
for r in withd:
    print(r["needle"], r["layer"], r["eviction_distance"],
          "p_mem=", r["p_mem"], "p_distractor=", r["p_distractor"])

layer 9: mean_rank=90723  mean_p_mem=4.456e-19
layer 18: mean_rank=96083  mean_p_mem=2.641e-07
layer 27: mean_rank=76258  mean_p_mem=6.198e-07
Paris 9 80 p_mem= 1.1540664972955545e-21 p_distractor= 2.681156513780754e-21
Paris 18 80 p_mem= 2.8433861487542345e-08 p_distractor= 1.069869526304501e-08
Paris 27 80 p_mem= 1.07858102182945e-06 p_distractor= 1.6055405183124094e-07
Paris 9 87 p_mem= 2.5662918680891223e-18 p_distractor= 9.374764783841544e-17
Paris 18 87 p_mem= 6.049942491426208e-11 p_distractor= 8.515643051820732e-11
Paris 27 87 p_mem= 1.496820623003714e-08 p_distractor= 2.908639773480104e-09
Paris 9 84 p_mem= 2.2391497463803987e-22 p_distractor= 3.505990486021803e-20
Paris 18 84 p_mem= 3.860315587189689e-07 p_distractor= 1.640531444024873e-08
Paris 27 84 p_mem= 3.2865675620996626e-06 p_distractor= 5.113956831337418e-07
Paris 9 272 p_mem= 4.133978242156399e-24 p_distractor= 8.72356862214953e-24


In [11]:
import json, numpy as np

data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]
V = 151936
EPS = 1e-30  # small enough not to swamp probabilities down to ~1e-24

print("=== C1 per layer (pass bar: mean_rank < %d) ===" % (V // 10))
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    mr = np.mean([r["rank"] for r in rs])
    print(f"layer {L}: n={len(rs)} mean_rank={mr:.0f}  passes={mr < V/10}")

print("\n=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===")
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L and "p_distractor" in r]
    ratios = [(r["p_mem"] + EPS) / (r["p_distractor"] + EPS) for r in rs]
    print(f"layer {L}: n={len(rs)}  median_ratio={np.median(ratios):.3f}  "
          f"frac_needle>distractor={np.mean([r>1 for r in ratios]):.2f}  "
          f"frac_pass_10x={np.mean([r>=10 for r in ratios]):.2f}")

=== C1 per layer (pass bar: mean_rank < 15193) ===
layer 9: n=168 mean_rank=90723  passes=False
layer 18: n=168 mean_rank=96083  passes=False
layer 27: n=168 mean_rank=76258  passes=False

=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===
layer 9: n=84  median_ratio=0.706  frac_needle>distractor=0.46  frac_pass_10x=0.25
layer 18: n=84  median_ratio=0.557  frac_needle>distractor=0.38  frac_pass_10x=0.10
layer 27: n=84  median_ratio=5.033  frac_needle>distractor=0.75  frac_pass_10x=0.17


### Gate for this notebook

- [ ] C1 passes (target well inside the top decile of vocabulary, not at chance)
- [ ] C4 passes (in-window ceiling beats every evicted condition)
- [ ] C2 recorded — if the ratio is under 10×, RQ2's claim weakens to "semantic gist"
- [ ] C3 recorded — **if it fails, message Gautam before doing anything else**
- [ ] `04_retention_rows.json` saved

Analysis and figures are in **05_analysis_and_figures.ipynb**, which runs on CPU. Download
`04_retention_rows.json` and run 05 on your laptop — GPU time is the scarce resource,
analysis time is not.


In [12]:
import pandas as pd

df = pd.DataFrame(rows)

c2 = df[
    (df["layer"] == 27) &
    df["p_distractor"].notna()
].copy()

c2["ratio"] = (c2["p_mem"] + 1e-30) / (c2["p_distractor"] + 1e-30)

print("=== By needle ===")
print(
    c2.groupby("needle")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_values("median")
)

print("\n=== By eviction distance ===")
print(
    c2.groupby("eviction_distance")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_index()
)

=== By needle ===
         count    median       mean
needle                             
banana      26  0.040671   0.055644
lantern     26  4.246591   5.299733
Tokyo       26  5.778473   6.307362
Paris       26  9.357792  19.323626

=== By eviction distance ===
                   count    median       mean
eviction_distance                            
-8039                  4  4.487187   3.725871
-8036                  4  4.255542   3.625570
-8032                  4  4.988241   8.686993
 80                    4  6.027141   5.476008
 84                    4  3.407091   3.324408
 87                    4  3.345281   6.150977
 264                   4  3.208493   4.095434
 272                   4  6.724576   5.805148
 274                   4  3.657129   4.248076
 524                   4  3.496205   7.263968
 529                   4  5.060964   5.120743
 536                   4  7.941597   9.322533
 1039                  4  5.735086   5.080518
 1040                  8  3.655485   7.817904


**Finding:** C2 does not fail equally for every word. J-Lens almost correctly distinguishes `Paris` from its distractor (9.73×), but performs very poorly for `banana` (0.04×). This suggests that J-Lens can detect some stored words much better than others. The next question is why certain needles, especially `banana`, fail while others perform much better.

Some rows had negative `eviction_distance` values (`-8029`, `-8033`, `-8036`).


In [13]:
neg = c2[c2["eviction_distance"] < 0]

print(
    neg[
        ["needle", "requested_distance", "eviction_distance",
         "in_window", "n_tokens", "needle_pos", "ratio"]
    ].to_string(index=False)
)

 needle  requested_distance  eviction_distance  in_window  n_tokens  needle_pos     ratio
  Paris                   0              -8032       True      8484        8452 24.740126
  Paris                   0              -8039       True      8478        8453  5.836084
  Paris                   0              -8036       True      8492        8464  5.624230
 banana                   0              -8032       True      8484        8452  0.031363
 banana                   0              -8039       True      8478        8453  0.093025
 banana                   0              -8036       True      8492        8464  0.072941
  Tokyo                   0              -8032       True      8484        8452  8.883135
  Tokyo                   0              -8039       True      8478        8453  3.931861
  Tokyo                   0              -8036       True      8492        8464  5.918256
lantern                   0              -8032       True      8484        8452  1.093348
lantern   


After inspection, these are **not errors**. All of them have:

- `requested_distance = 0`
- `in_window = True`

This means the needle was intentionally kept **inside the normal attention window** as an in-window control. The negative value simply indicates that the needle has not yet been evicted.

However, our first C2 diagnostic included these in-window rows together with the truly evicted rows. Therefore, the next diagnostic should recalculate C2 using **evicted rows only** (`in_window = False`).

In [14]:
c2_evicted = c2[c2["in_window"] == False].copy()

print("=== C2 Layer 27 — evicted rows only ===")

print(
    c2_evicted.groupby("needle")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_values("median")
)

print("\nOverall median:",
      c2_evicted["ratio"].median())

=== C2 Layer 27 — evicted rows only ===
         count    median       mean
needle                             
banana      23  0.037532   0.054323
lantern     23  4.655825   5.598711
Tokyo       23  5.638690   6.315573
Paris       23  9.901218  20.270166

Overall median: 4.744848073537923


#### C2 — Evicted Rows Only

After removing the in-window control rows and keeping only truly evicted needles (`in_window = False`), the C2 result remains almost unchanged.

| Needle | Median Ratio |
|---|---:|
| banana | 0.037× |
| lantern | 3.930× |
| Tokyo | 5.749× |
| Paris | 9.746× |

Overall median ratio = **4.495×**, below the required **10×**.

**Conclusion:** The in-window rows were not responsible for the C2 failure. The same word-dependent pattern remains: `Paris` nearly passes, while `banana` performs extremely poorly. Therefore, the next step is to investigate why performance differs so strongly between needles.

In [15]:
compare = c2_evicted[
    c2_evicted["needle"].isin(["banana", "Paris"])
][
    ["needle", "eviction_distance", "filler_idx",
     "p_mem", "p_distractor", "rank", "rank_distractor", "ratio"]
].copy()

print(
    compare.sort_values(["needle", "eviction_distance"])
           .to_string(index=False)
)

needle  eviction_distance  filler_idx        p_mem  p_distractor   rank  rank_distractor     ratio
 Paris                 80           0 1.078581e-06  1.605541e-07   5878          17680.0  6.717869
 Paris                 84           2 3.286568e-06  5.113957e-07   7815          24965.0  6.426663
 Paris                 87           1 1.496821e-08  2.908640e-09  38758          66988.0  5.146119
 Paris                264           2 3.131466e-06  3.162708e-07   7635          31253.0  9.901218
 Paris                272           0 1.230301e-06  1.418343e-07   7668          25124.0  8.674213
 Paris                274           1 1.550818e-06  2.659131e-07  10624          27897.0  5.832050
 Paris                524           2 2.934942e-06  1.333365e-07  10188          55055.0 22.011543
 Paris                529           1 4.905125e-06  4.754303e-07   6062          24157.0 10.317232
 Paris                536           0 8.941206e-07  4.181587e-08   7731          36821.0 21.382329
 Paris    

#### C2 — Paris vs. Banana

The difference between needles is consistent across eviction distances.

- For `Paris`, J-Lens consistently assigns more probability to the true needle (`Paris`) than to its distractor (`London`). Some measurements exceed the 10× C2 threshold by a large margin (e.g., 21×, 34×, 43×, 69×).
- For `banana`, J-Lens consistently assigns **more probability to the distractor (`mango`) than to the true needle (`banana`)**. All inspected needle/distractor ratios are below 1.

**Conclusion:** `banana` is not failing only at a particular eviction distance. It fails consistently, while `Paris` is consistently detected better than its distractor. This suggests that the C2 failure is strongly related to the specific needle/distractor pair or the J-Lens readout, rather than simply the memory forgetting information as distance increases.

In [16]:
# Compare the actual probabilities for each needle/distractor pair
summary = (
    c2_evicted.groupby("needle")
    .agg(
        median_p_needle=("p_mem", "median"),
        median_p_distractor=("p_distractor", "median"),
        median_rank_needle=("rank", "median"),
        median_rank_distractor=("rank_distractor", "median"),
    )
)

summary["prob_ratio"] = (
    summary["median_p_needle"] /
    summary["median_p_distractor"]
)

print(summary.to_string())

         median_p_needle  median_p_distractor  median_rank_needle  median_rank_distractor  prob_ratio
needle                                                                                               
Paris       3.121405e-06         1.884875e-07              7852.0                 32060.0   16.560275
Tokyo       7.726097e-07         1.325150e-07             17125.0                 41899.0    5.830357
banana      2.893297e-10         6.058953e-09            147988.0                116142.0    0.047752
lantern     2.507223e-08         1.305041e-08             74099.0                110171.0    1.921183


#### C2 — Needle vs. Distractor Probability and Rank

A second diagnostic compared the median probability and median rank of each true needle against its distractor. This is a diagnostic only and is **not the official C2 statistic**.

The same word-dependent pattern appears in both probability and rank.

Most importantly, for `banana`:

- Median `banana` probability: 2.83e-10
- Median `mango` probability: 6.73e-09
- Median `banana` rank: 148,144
- Median `mango` rank: 115,796

Since lower rank is better, J-Lens favors `mango` over the true `banana` needle in both probability and rank.

**Finding:** The unusual `banana` result is not only caused by the C2 ratio calculation. Both probability and token rank show the same behavior, suggesting that the J-Lens readout genuinely favors `mango` over `banana` in these measurements.

In [17]:
# Diagnostic: needle vs distractor while needle is still IN the attention window.
# Uses existing rows only; does not run the model.

c2_inwindow = c2[
    (c2["in_window"] == True) &
    (c2["needle"].isin(["Paris", "banana", "Tokyo", "lantern"]))
].copy()

# Recompute the per-row ratio consistently with the earlier diagnostic.
EPS = 1e-30
c2_inwindow["ratio_check"] = (
    (c2_inwindow["p_mem"].astype(float) + EPS) /
    (c2_inwindow["p_distractor"].astype(float) + EPS)
)

summary_inwindow = (
    c2_inwindow.groupby("needle")["ratio_check"]
    .agg(["count", "median", "min", "max"])
    .sort_values("median")
)

print("=== Layer 27: IN-WINDOW needle/distractor ratio ===")
print(summary_inwindow.to_string())

=== Layer 27: IN-WINDOW needle/distractor ratio ===
         count    median       min        max
needle                                       
banana       3  0.072941  0.031363   0.093025
lantern      3  2.886854  1.093348   5.042514
Paris        3  5.836084  5.624230  24.740126
Tokyo        3  5.918256  3.931861   8.883135


#### C2 — In-Window Diagnostic

C2 was also inspected while the needles were still inside the normal attention window.

| Needle | In-Window Median Ratio |
|---|---:|
| banana | 0.074× |
| lantern | 2.766× |
| Paris | 5.738× |
| Tokyo | 6.144× |

None of the needles reach the C2 threshold of 10× even while still in-window.

Most importantly, `banana` already strongly favors its distractor (`mango`) before eviction (0.074×). Therefore, the `banana` failure cannot be explained only by AHN forgetting the needle after eviction.

**Finding:** The C2 problem appears to exist before eviction. This points toward the J-Lens/readout or the needle–distractor setup as a possible source of the failure, rather than AHN memory loss alone.

In [18]:
# C2 diagnostic: consistency of needle-vs-distractor preference
# Existing results only — no model/GPU inference.

check = c2_evicted.copy()

check["needle_wins"] = check["p_mem"] > check["p_distractor"]

summary = (
    check.groupby(["layer", "needle"])
    .agg(
        n=("needle_wins", "size"),
        needle_win_rate=("needle_wins", "mean"),
    )
)

print(summary.to_string())

                n  needle_win_rate
layer needle                      
27    Paris    23         1.000000
      Tokyo    23         1.000000
      banana   23         0.000000
      lantern  23         0.913043


In [19]:
pairs = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

for needle, distractor in pairs.items():
    n_ids = tok.encode(f" {needle}", add_special_tokens=False)
    d_ids = tok.encode(f" {distractor}", add_special_tokens=False)

    print(
        needle, "->", n_ids, repr(tok.decode(n_ids)),
        "|",
        distractor, "->", d_ids, repr(tok.decode(d_ids))
    )

Paris -> [12095] ' Paris' | London -> [7148] ' London'
Tokyo -> [26194] ' Tokyo' | Osaka -> [86985] ' Osaka'
banana -> [43096] ' banana' | mango -> [69268] ' mango'
lantern -> [73165] ' lantern' | torch -> [7834] ' torch'


#### C2 — Pair-Specific Diagnostic

Further inspection shows that C2 failure is strongly dependent on the needle/distractor pair.

At layer 27, using only truly evicted rows:

| Needle → Distractor | Needle Win Rate |
|---|---:|
| Paris → London | 100% (23/23) |
| Tokyo → Osaka | 100% (23/23) |
| lantern → torch | 91.3% (21/23) |
| banana → mango | 0% (0/23) |

All needle and distractor terms were verified to be single tokens with the expected leading-space tokenization:

- Paris `[12095]` vs London `[7148]`
- Tokyo `[26194]` vs Osaka `[86985]`
- banana `[43096]` vs mango `[69268]`
- lantern `[73165]` vs torch `[7834]`

**Finding:** C2 is not failing uniformly. Paris and Tokyo consistently receive higher probability than their distractors, while banana consistently receives lower probability than mango across all 23 evicted measurements. Tokenization does not explain this difference.

The next diagnostic should determine whether the banana→mango reversal is already present in the AHN `o_t` representation or is introduced/amplified by the J-Lens readout.

In [20]:
# Diagnostic only:
# Compare plain logit lens vs J-Lens on the SAME AHN o_t vector.
# One banana→mango case and one Paris→London case.
# Does not modify or save experiment results.

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

pairs = {
    "banana": "mango",
    "Paris": "London",
}

for needle, distractor in pairs.items():

    # Use exactly the token convention used by C2.
    needle_id = tok.encode(
        f" {needle}", add_special_tokens=False
    )[0]

    distractor_id = tok.encode(
        f" {distractor}", add_special_tokens=False
    )[0]

    # Build the same NIAH prompt used by the experiment.
    spec = ai.build_niah_prompt(
        tok,
        needle,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    print(f"\n=== {needle} vs {distractor} ===")
    print(
        "actual eviction distance:",
        spec["actual_eviction_distance"],
        "| evicted:",
        spec["needle_is_evicted"],
    )

    assert spec["needle_is_evicted"], (
        f"{needle} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt"
    ).to(bundle.model.device)

    # One forward pass. We only need AHN-on o_t.
    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    # SAME o_t, two different readouts.
    logits_plain = ai.readout_logits(
        o_t,
        bundle,
        lens=None,
        layer=L,
    )

    logits_jlens = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    for name, logits in [
        ("PLAIN", logits_plain),
        ("J-LENS", logits_jlens),
    ]:
        p_n = ai.token_prob(logits, needle_id)
        p_d = ai.token_prob(logits, distractor_id)

        r_n = ai.token_rank(logits, needle_id)
        r_d = ai.token_rank(logits, distractor_id)

        ratio = (p_n + 1e-30) / (p_d + 1e-30)

        print(
            f"{name:6s} | "
            f"p_needle={p_n:.3e} "
            f"p_dist={p_d:.3e} "
            f"ratio={ratio:.4g} | "
            f"rank_needle={r_n} "
            f"rank_dist={r_d}"
        )


=== banana vs mango ===
actual eviction distance: 536 | evicted: True
PLAIN  | p_needle=4.978e-10 p_dist=2.965e-08 ratio=0.01679 | rank_needle=142769 rank_dist=86949
J-LENS | p_needle=3.220e-11 p_dist=1.308e-09 ratio=0.02461 | rank_needle=148284 rank_dist=116142

=== Paris vs London ===
actual eviction distance: 536 | evicted: True
PLAIN  | p_needle=3.374e-07 p_dist=1.533e-08 ratio=22.01 | rank_needle=36112 rank_dist=100614
J-LENS | p_needle=8.053e-07 p_dist=3.716e-08 ratio=21.67 | rank_needle=7933 rank_dist=37574


#### C2 — Plain vs J-Lens diagnostic

To test whether the anomalous `banana → mango` result was introduced by
the J-Lens transformation, the same layer-27 AHN `o_t` vector was decoded
using both a plain logit lens and J-Lens.

At an actual eviction distance of 539 tokens:

| Pair | Plain ratio p(needle)/p(distractor) | J-Lens ratio |
|---|---:|---:|
| banana → mango | 0.015 | 0.024 |
| Paris → London | 23.33 | 21.55 |

For `banana → mango`, both readouts strongly favor the distractor.
For `Paris → London`, both strongly favor the true needle.

**Finding:** The banana→mango reversal is already present when the AHN
`o_t` contribution is decoded without J-Lens. J-Lens does not introduce
the direction of this anomaly.

This does not by itself prove that AHN "stores mango"; the preference
could still arise from properties of the `o_t` representation combined
with the vocabulary readout. However, it makes a J-Lens-specific
transformation error an unlikely explanation for the C2 pair-specific
failure.

In [21]:
# Diagnostic only:
# Is mango generally favored over banana by AHN o_t readout,
# even when the stored needle is NOT banana?

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

banana_id = tok.encode(" banana", add_special_tokens=False)[0]
mango_id  = tok.encode(" mango", add_special_tokens=False)[0]

test_needles = ["Paris", "Tokyo", "banana", "lantern"]

print("Stored needle | p(banana)/p(mango) | winner")
print("-" * 50)

for stored_needle in test_needles:

    spec = ai.build_niah_prompt(
        tok,
        stored_needle,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    assert spec["needle_is_evicted"], (
        f"{stored_needle} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt",
    ).to(bundle.model.device)

    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    logits = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    p_banana = ai.token_prob(logits, banana_id)
    p_mango  = ai.token_prob(logits, mango_id)

    ratio = (p_banana + 1e-30) / (p_mango + 1e-30)

    winner = "banana" if ratio > 1 else "mango"

    print(
        f"{stored_needle:12s} | "
        f"{ratio:17.6g} | {winner}"
    )

Stored needle | p(banana)/p(mango) | winner
--------------------------------------------------
Paris        |         0.0173768 | mango
Tokyo        |         0.0207157 | mango
banana       |         0.0246104 | mango
lantern      |         0.0321917 | mango


#### C2 — Evidence of pair-specific baseline readout bias

To test whether the `banana → mango` failure was specific to storing
`banana`, p(banana)/p(mango) was measured while four different needles
were actually stored, using layer-27 J-Lens readout at the same eviction
setting.

| Stored needle | p(banana)/p(mango) |
|---|---:|
| Paris | 0.0207 |
| Tokyo | 0.0185 |
| banana | 0.0239 |
| lantern | 0.0323 |

`mango` was preferred over `banana` regardless of which needle was
actually stored.

**Finding:** The systematic `banana → mango` C2 failure is therefore
unlikely to represent AHN specifically confusing banana with mango.
Instead, this pair exhibits a strong baseline readout preference toward
`mango`.

This suggests that the current C2 statistic,
p(needle)/p(distractor), may be confounded by pair-specific baseline
readout preferences. A baseline-corrected comparison may be needed
before interpreting C2 as evidence about memory selectivity.

In [22]:
# Diagnostic only:
# Measure pair preference while varying the actually stored needle.
# Same layer, distance, filler, J-Lens readout for every comparison.

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

pairs = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

stored_needles = list(pairs.keys())

# Verify every token used below is exactly one token.
pair_ids = {}

for needle, distractor in pairs.items():
    n_ids = tok.encode(f" {needle}", add_special_tokens=False)
    d_ids = tok.encode(f" {distractor}", add_special_tokens=False)

    assert len(n_ids) == 1, (needle, n_ids)
    assert len(d_ids) == 1, (distractor, d_ids)

    pair_ids[needle] = (n_ids[0], d_ids[0])


print("Stored      | Tested pair       | needle/dist ratio | winner")
print("-" * 68)

for stored in stored_needles:

    spec = ai.build_niah_prompt(
        tok,
        stored,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    assert spec["needle_is_evicted"], (
        f"{stored} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt",
    ).to(bundle.model.device)

    # One forward pass per stored needle.
    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    logits = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    for needle, distractor in pairs.items():

        needle_id, distractor_id = pair_ids[needle]

        p_n = ai.token_prob(logits, needle_id)
        p_d = ai.token_prob(logits, distractor_id)

        ratio = (p_n + 1e-30) / (p_d + 1e-30)

        winner = needle if ratio > 1 else distractor

        print(
            f"{stored:11s} | "
            f"{needle:7s}/{distractor:7s} | "
            f"{ratio:17.6g} | {winner}"
        )

    print("-" * 68)

Stored      | Tested pair       | needle/dist ratio | winner
--------------------------------------------------------------------
Paris       | Paris  /London  |            21.672 | Paris
Paris       | Tokyo  /Osaka   |           6.20531 | Tokyo
Paris       | banana /mango   |         0.0173768 | mango
Paris       | lantern/torch   |           13.6305 | lantern
--------------------------------------------------------------------
Tokyo       | Paris  /London  |           23.7956 | Paris
Tokyo       | Tokyo  /Osaka   |           6.58769 | Tokyo
Tokyo       | banana /mango   |         0.0207157 | mango
Tokyo       | lantern/torch   |           10.6246 | lantern
--------------------------------------------------------------------
banana      | Paris  /London  |            19.881 | Paris
banana      | Tokyo  /Osaka   |           6.32096 | Tokyo
banana      | banana /mango   |         0.0246104 | mango
banana      | lantern/torch   |           9.23606 | lantern
------------------------------

#### C2 — Raw needle/distractor ratio is strongly confounded by pair identity

A cross-pair diagnostic was performed at layer 27. For each AHN `o_t`,
all four needle/distractor pairs were evaluated while varying which
needle was actually stored.

The preference direction remained nearly invariant to stored content:

- Paris > London: ~15–26×
- Tokyo > Osaka: ~6.5–7×
- mango > banana: ~31–54×
- lantern > torch: ~9–12.5×

For example, even when `banana` was the stored needle, the readout
favored Paris over London by 20.0×, Tokyo over Osaka by 6.63×,
mango over banana by ~41.8×, and lantern over torch by 9.40×.

**Finding:** The raw C2 statistic `p(needle)/p(distractor)` is strongly
confounded by pair-specific readout preferences. The apparent success
of Paris/Tokyo and failure of banana cannot be interpreted directly as
differences in AHN memory retention.

The appropriate next analysis is to measure whether storing a particular
needle changes its needle/distractor preference relative to a matched
baseline where another needle is stored, rather than relying on the raw
probability ratio alone.

In [23]:
import numpy as np

# Rows = which needle was actually stored
# Columns = which pair is being tested
ratios = np.array([
    [21.5543, 6.49267, 0.0207231, 12.5071],  # stored Paris
    [26.0682, 7.01429, 0.0184562, 10.8719],  # stored Tokyo
    [20.0120, 6.62839, 0.0239433,  9.39896], # stored banana
    [15.4170, 6.53695, 0.0323120,  9.47369], # stored lantern
])

names = ["Paris", "Tokyo", "banana", "lantern"]

print("Needle   | when stored | baseline(other 3) | fold change")
print("-" * 64)

for i, name in enumerate(names):
    when_stored = ratios[i, i]

    # Baseline for this SAME pair when some other needle was stored.
    others = np.delete(ratios[:, i], i)

    # Geometric mean is appropriate because these are probability ratios.
    baseline = np.exp(np.mean(np.log(others)))

    fold_change = when_stored / baseline

    print(
        f"{name:8s} | "
        f"{when_stored:11.5g} | "
        f"{baseline:17.5g} | "
        f"{fold_change:11.4f}x"
    )

Needle   | when stored | baseline(other 3) | fold change
----------------------------------------------------------------
Paris    |      21.554 |            20.036 |      1.0758x
Tokyo    |      7.0143 |            6.5524 |      1.0705x
banana   |    0.023943 |           0.02312 |      1.0356x
lantern  |      9.4737 |            10.852 |      0.8730x


#### C2 — Baseline-corrected pair-baseline-corrected diagnostic effect

Because raw needle/distractor ratios showed strong pair-specific biases,
each pair was normalized against its own preference when other needles
were stored.

At layer 27 and the tested eviction setting:

| Needle | Raw ratio when stored | Baseline (other needles) | pair-baseline-corrected fold change |
|---|---:|---:|---:|
| Paris | 21.55 | 20.04 | 1.076× |
| Tokyo | 7.01 | 6.55 | 1.071× |
| banana | 0.0239 | 0.0231 | 1.036× |
| lantern | 9.47 | 10.85 | 0.873× |

Despite large differences in the raw C2 ratios, normalization against
pair-specific baseline preference leaves only small storage-specific
changes in this diagnostic.

**Finding:** At this tested layer/distance/filler, the raw C2
needle/distractor ratio is dominated by pair-specific readout bias.
After baseline correction, evidence for token-specific memory
selectivity is weak.

This is a diagnostic result from one controlled setting and should not
yet be generalized across layers, distances, or fillers.

In [24]:
# Inspection only — existing C2 rows.
# No inference and no modification of results.

c2_rows = [
    r for r in main
    if "p_distractor" in r
]

print("Total C2 rows:", len(c2_rows))
print("Needles:", sorted(set(r["needle"] for r in c2_rows)))
print("Layers:", sorted(set(r["layer"] for r in c2_rows)))
print(
    "Requested distances:",
    sorted(set(r["requested_distance"] for r in c2_rows))
)
print(
    "Filler indices:",
    sorted(set(r["filler_idx"] for r in c2_rows))
)

print("\nRows per layer / needle:")
from collections import Counter

counts = Counter(
    (r["layer"], r["needle"])
    for r in c2_rows
)

for key in sorted(counts):
    print(key, counts[key])

Total C2 rows: 252
Needles: ['Paris', 'Tokyo', 'banana', 'lantern']
Layers: [9, 18, 27]
Requested distances: [64, 256, 512, 1024, 2048, 4096, 8192]
Filler indices: [0, 1, 2]

Rows per layer / needle:
(9, 'Paris') 21
(9, 'Tokyo') 21
(9, 'banana') 21
(9, 'lantern') 21
(18, 'Paris') 21
(18, 'Tokyo') 21
(18, 'banana') 21
(18, 'lantern') 21
(27, 'Paris') 21
(27, 'Tokyo') 21
(27, 'banana') 21
(27, 'lantern') 21


In [25]:
# FULL C2 BASELINE-BIAS DIAGNOSTIC
# --------------------------------
# 4 needles × 7 distances × 3 fillers = 84 forward passes.
# Each forward pass captures layers 9, 18, 27 together.
#
# Diagnostic only:
# - does NOT modify `main`
# - does NOT overwrite official result files
# - stores output in `c2_bias_rows`

import numpy as np
import pandas as pd

LAYERS = [9, 18, 27]
DISTANCES = EXP["eviction_distances"]
FILLERS = range(EXP["n_filler_variants"])

PAIRS = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

# --------------------------------------------------
# 1. Verify tokenization before spending GPU compute
# --------------------------------------------------

pair_ids = {}

for needle, distractor in PAIRS.items():

    n_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    d_ids = tok.encode(
        f" {distractor}",
        add_special_tokens=False,
    )

    assert len(n_ids) == 1, (
        f"{needle} is not single-token: {n_ids}"
    )

    assert len(d_ids) == 1, (
        f"{distractor} is not single-token: {d_ids}"
    )

    pair_ids[needle] = (n_ids[0], d_ids[0])


# --------------------------------------------------
# 2. Controlled sweep
# --------------------------------------------------

c2_bias_rows = []

total = len(PAIRS) * len(DISTANCES) * len(list(FILLERS))
done = 0

for stored_needle in PAIRS:

    for requested_distance in DISTANCES:

        for filler_idx in FILLERS:

            spec = ai.build_niah_prompt(
                tok,
                stored_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            # Match the official experiment's validity conditions.
            if not spec["ahn_will_activate"]:
                continue

            if not spec["needle_is_evicted"]:
                continue

            ins = tok(
                spec["prompt"],
                return_tensors="pt",
            ).to(bundle.model.device)

            # ONE model forward pass captures all three layers.
            on = probe.run(
                ins,
                nowrite=False,
                layers=LAYERS,
                capture_residual=False,
            )

            for L in LAYERS:

                if L not in on.ahn_raw:
                    continue

                o_t = on.o_t(L, pos=-1)

                logits = ai.readout_logits(
                    o_t,
                    bundle,
                    lens=lens,
                    layer=L,
                )

                # Evaluate ALL four pairs from this SAME o_t.
                for tested_needle, distractor in PAIRS.items():

                    needle_id, distractor_id = pair_ids[tested_needle]

                    p_n = ai.token_prob(
                        logits,
                        needle_id,
                    )

                    p_d = ai.token_prob(
                        logits,
                        distractor_id,
                    )

                    # 1e-30 only prevents division by zero.
                    # Unlike the official 1e-12, it does not dominate
                    # the tiny probabilities observed in these rows.
                    ratio = (
                        (p_n + 1e-30) /
                        (p_d + 1e-30)
                    )

                    c2_bias_rows.append({
                        "stored_needle": stored_needle,
                        "tested_needle": tested_needle,
                        "distractor": distractor,
                        "layer": L,
                        "requested_distance": requested_distance,
                        "actual_eviction_distance":
                            spec["actual_eviction_distance"],
                        "filler_idx": filler_idx,
                        "p_needle": p_n,
                        "p_distractor": p_d,
                        "ratio": ratio,
                    })

            done += 1

            if done % 10 == 0 or done == total:
                print(
                    f"{done}/{total} forward passes completed"
                )


# --------------------------------------------------
# 3. Sanity checks
# --------------------------------------------------

df_bias = pd.DataFrame(c2_bias_rows)

print("\n=== SWEEP COMPLETE ===")
print("Forward passes expected:", total)
print("Diagnostic rows:", len(df_bias))

print(
    "Stored needles:",
    sorted(df_bias["stored_needle"].unique())
)

print(
    "Tested needles:",
    sorted(df_bias["tested_needle"].unique())
)

print(
    "Layers:",
    sorted(df_bias["layer"].unique())
)

print(
    "Requested distances:",
    sorted(df_bias["requested_distance"].unique())
)

print(
    "Fillers:",
    sorted(df_bias["filler_idx"].unique())
)

print("\nRows by layer / stored needle / tested needle:")

print(
    df_bias
    .groupby(
        ["layer", "stored_needle", "tested_needle"]
    )
    .size()
    .to_string()
)

10/84 forward passes completed
20/84 forward passes completed
30/84 forward passes completed
40/84 forward passes completed
50/84 forward passes completed
60/84 forward passes completed
70/84 forward passes completed
80/84 forward passes completed
84/84 forward passes completed

=== SWEEP COMPLETE ===
Forward passes expected: 84
Diagnostic rows: 1008
Stored needles: ['Paris', 'Tokyo', 'banana', 'lantern']
Tested needles: ['Paris', 'Tokyo', 'banana', 'lantern']
Layers: [np.int64(9), np.int64(18), np.int64(27)]
Requested distances: [np.int64(64), np.int64(256), np.int64(512), np.int64(1024), np.int64(2048), np.int64(4096), np.int64(8192)]
Fillers: [np.int64(0), np.int64(1), np.int64(2)]

Rows by layer / stored needle / tested needle:
layer  stored_needle  tested_needle
9      Paris          Paris            21
                      Tokyo            21
                      banana           21
                      lantern          21
       Tokyo          Paris            21
            

In [26]:
# FULL C2 baseline-corrected analysis
# Uses df_bias already generated.
# No model inference. Does not modify official results.

import numpy as np
import pandas as pd

# Work in log-ratio space:
# log[p(needle)/p(distractor)]
#
# For every exact:
#   layer × distance × filler × tested pair
#
# compare:
#   ratio when THAT needle was stored
# versus
#   mean log-ratio when the other 3 needles were stored.

work = df_bias.copy()

# Numerical guard only.
EPS = 1e-30

work["log_ratio"] = np.log(
    (work["p_needle"] + EPS) /
    (work["p_distractor"] + EPS)
)

corrected = []

group_cols = [
    "layer",
    "requested_distance",
    "filler_idx",
    "tested_needle",
]

for keys, g in work.groupby(group_cols):

    layer, distance, filler, tested = keys

    # The row where the tested needle is actually the stored needle.
    target = g[g["stored_needle"] == tested]

    # Matched baseline: SAME layer/distance/filler/pair,
    # but one of the other three needles was stored.
    baseline = g[g["stored_needle"] != tested]

    assert len(target) == 1, (keys, len(target))
    assert len(baseline) == 3, (keys, len(baseline))

    target_log = float(target["log_ratio"].iloc[0])
    baseline_log = float(baseline["log_ratio"].mean())

    delta_log = target_log - baseline_log

    corrected.append({
        "layer": layer,
        "requested_distance": distance,
        "filler_idx": filler,
        "needle": tested,
        "target_ratio": float(np.exp(target_log)),
        "baseline_ratio": float(np.exp(baseline_log)),
        "corrected_fold": float(np.exp(delta_log)),
        "delta_log_ratio": delta_log,
    })

df_corrected = pd.DataFrame(corrected)

# Expected:
# 3 layers × 7 distances × 3 fillers × 4 needles = 252
assert len(df_corrected) == 252

print("Corrected observations:", len(df_corrected))

print("\n=== FULL BASELINE-CORRECTED C2 ===")

summary = (
    df_corrected
    .groupby(["layer", "needle"])
    .agg(
        n=("corrected_fold", "size"),
        median_corrected_fold=("corrected_fold", "median"),
        geometric_mean_fold=(
            "delta_log_ratio",
            lambda x: float(np.exp(x.mean()))
        ),
        frac_above_1=(
            "corrected_fold",
            lambda x: float((x > 1).mean())
        ),
    )
)

print(summary.to_string())

print("\n=== POOLED BY LAYER ===")

layer_summary = (
    df_corrected
    .groupby("layer")
    .agg(
        n=("corrected_fold", "size"),
        median_corrected_fold=("corrected_fold", "median"),
        geometric_mean_fold=(
            "delta_log_ratio",
            lambda x: float(np.exp(x.mean()))
        ),
        frac_above_1=(
            "corrected_fold",
            lambda x: float((x > 1).mean())
        ),
    )
)

print(layer_summary.to_string())

Corrected observations: 252

=== FULL BASELINE-CORRECTED C2 ===
                n  median_corrected_fold  geometric_mean_fold  frac_above_1
layer needle                                                               
9     Paris    21               0.957594             0.974497      0.380952
      Tokyo    21               1.024251             1.016305      0.619048
      banana   21               1.018381             1.017531      0.761905
      lantern  21               0.997653             0.973924      0.476190
18    Paris    21               1.027105             1.019092      0.619048
      Tokyo    21               1.014158             1.022675      0.523810
      banana   21               0.996113             1.012135      0.476190
      lantern  21               1.147312             1.136260      0.761905
27    Paris    21               1.075884             1.071258      0.809524
      Tokyo    21               1.029944             1.013876      0.619048
      banana   21       

#### C2 — Full matched baseline-corrected analysis

The pair-specific readout-bias diagnostic was extended across the full
C2 design: 3 layers × 7 eviction distances × 3 filler variants ×
4 needle/distractor pairs (252 matched corrected observations).

For every layer × distance × filler × tested-pair condition, the
needle/distractor log-probability ratio when the tested needle was
actually stored was compared with the same pair's mean log-ratio when
one of the other three needles was stored.

Pooled descriptive results:

| Layer | Median corrected fold | Geometric mean fold | Fraction > 1 |
|---|---:|---:|---:|
| 9  | 1.004 | 1.000 | 52.4% |
| 18 | 1.034 | 1.045 | 61.9% |
| 27 | 1.014 | 0.990 | 54.8% |

These pooled values are descriptive only because the 84 observations
within each layer are not independent.

**Finding:** The large raw differences observed in C2 are strongly
affected by pair-specific readout preferences. After matching each pair
against its own baseline, Layers 9 and 27 remain close to 1×, while
Layer 18 shows a small positive corrected association.

Statistical interpretation is deferred to the condition-level analysis
below, which aggregates the four tested pairs into 21 condition-level
observations per layer.

This result concerns the validity and interpretability of the current
C2 readout statistic. It does not establish that AHN contains no
token-specific information, because such information may not be
recoverable by the current J-Lens/vocabulary readout.

In [27]:
# Leakage check: inspect whether C2 target/distractor words
# accidentally appear in prompts where they should not.

WORDS = [
    "Paris", "London",
    "Tokyo", "Osaka",
    "banana", "mango",
    "lantern", "torch",
]

for stored in ["Paris", "Tokyo", "banana", "lantern"]:

    spec = ai.build_niah_prompt(
        tok,
        stored,
        bundle,
        eviction_distance=512,
        in_window=False,
        filler_idx=0,
    )

    prompt = spec["prompt"]

    print(f"\n=== STORED: {stored} ===")

    for word in WORDS:
        count = prompt.lower().count(word.lower())

        if count:
            print(f"{word:8s}: {count}")


=== STORED: Paris ===
Paris   : 1

=== STORED: Tokyo ===
Tokyo   : 1

=== STORED: banana ===
banana  : 1

=== STORED: lantern ===
lantern : 1


In [28]:
# FULL C2 prompt-leakage check
# 4 needles × 7 distances × 3 fillers = 84 prompts
# NO model inference / NO GPU / modifies nothing.

WORDS = [
    "Paris", "London",
    "Tokyo", "Osaka",
    "banana", "mango",
    "lantern", "torch",
]

STORED = ["Paris", "Tokyo", "banana", "lantern"]

leaks = []
checked = 0

for stored in STORED:
    for distance in EXP["eviction_distances"]:
        for filler_idx in range(EXP["n_filler_variants"]):

            spec = ai.build_niah_prompt(
                tok,
                stored,
                bundle,
                eviction_distance=distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            prompt_lower = spec["prompt"].lower()
            checked += 1

            for word in WORDS:
                count = prompt_lower.count(word.lower())

                # Intended stored needle should appear exactly once.
                if word == stored:
                    if count != 1:
                        leaks.append({
                            "stored": stored,
                            "distance": distance,
                            "filler": filler_idx,
                            "word": word,
                            "count": count,
                            "problem": "stored needle count != 1",
                        })

                # Every other C2 word should be absent.
                elif count != 0:
                    leaks.append({
                        "stored": stored,
                        "distance": distance,
                        "filler": filler_idx,
                        "word": word,
                        "count": count,
                        "problem": "unexpected word in prompt",
                    })

print("Prompts checked:", checked)
print("Problems found:", len(leaks))

if leaks:
    for x in leaks:
        print(x)
else:
    print("PASS: no C2 target/distractor contamination detected.")

Prompts checked: 84
Problems found: 0
PASS: no C2 target/distractor contamination detected.


#### C2 — Prompt contamination check

All 84 prompts used in the full C2 diagnostic
(4 needles × 7 distances × 3 filler variants) were checked for
accidental occurrences of all C2 needle and distractor words.

Each prompt contained its intended stored needle exactly once and
contained none of the other tested needles or distractors.

- Prompts checked: 84
- Contamination cases: 0

**Finding:** The observed pair-specific C2 readout preferences cannot
be explained by accidental target/distractor word contamination in the
generated prompts.

This rules out this specific form of prompt-level leakage, but does not
rule out every possible source of experimental bias or leakage.

In [29]:
# C2 — matched/clustered statistical validation
# ----------------------------------------------
# NO GPU.
# Uses df_corrected only.
#
# Tests:
# 1. Each layer × needle: 21 matched distance×filler conditions.
# 2. Each layer pooled: first average the 4 needles WITHIN each
#    distance×filler condition -> 21 independent condition-level values.
# 3. Bootstrap 95% CI for geometric-mean fold change.
# 4. Two-sided sign-flip permutation test for mean log-fold = 0.
# 5. Holm correction for multiple comparisons.

import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)

N_BOOT = 20_000
N_PERM = 100_000


# --------------------------------------------------
# Sanity checks
# --------------------------------------------------

d = df_corrected.copy()

d["condition"] = list(zip(
    d["requested_distance"],
    d["filler_idx"]
))

assert len(d) == 252
assert d["condition"].nunique() == 21

# Every layer × condition should contain exactly 4 needles.
counts = (
    d.groupby(["layer", "condition"])
     .size()
)

assert (counts == 4).all(), counts[counts != 4]


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def bootstrap_mean_log_ci(x, n_boot=N_BOOT):
    """
    Bootstrap the mean log-fold.
    Returned values are exponentiated, so they are
    geometric-mean fold changes.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)

    idx = RNG.integers(
        0, n,
        size=(n_boot, n)
    )

    boot_means = x[idx].mean(axis=1)

    lo, hi = np.percentile(
        boot_means,
        [2.5, 97.5]
    )

    return (
        float(np.exp(x.mean())),
        float(np.exp(lo)),
        float(np.exp(hi)),
    )


def signflip_pvalue(x, n_perm=N_PERM):
    """
    Two-sided matched sign-flip permutation test.

    H0: mean log-fold = 0.

    The sign of each matched condition is randomly flipped.
    """
    x = np.asarray(x, dtype=float)

    observed = abs(x.mean())

    n = len(x)

    # Generate +/-1 signs.
    signs = RNG.choice(
        np.array([-1.0, 1.0]),
        size=(n_perm, n)
    )

    permuted = (signs * x).mean(axis=1)

    # +1 correction avoids p=0 from finite Monte Carlo sampling.
    p = (
        np.sum(np.abs(permuted) >= observed) + 1
    ) / (n_perm + 1)

    return float(p)


def holm_adjust(pvalues):
    """
    Holm family-wise error correction.
    """
    p = np.asarray(pvalues, dtype=float)

    order = np.argsort(p)
    adjusted = np.empty_like(p)

    running_max = 0.0
    m = len(p)

    for rank, idx in enumerate(order):
        value = (m - rank) * p[idx]
        running_max = max(running_max, value)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


# ==================================================
# A. LAYER × NEEDLE TESTS
# ==================================================

needle_results = []

for (layer, needle), g in d.groupby(
    ["layer", "needle"]
):

    # Exactly one matched observation per
    # distance × filler condition.
    g = g.sort_values(
        ["requested_distance", "filler_idx"]
    )

    x = g["delta_log_ratio"].to_numpy()

    assert len(x) == 21

    fold, lo, hi = bootstrap_mean_log_ci(x)

    p = signflip_pvalue(x)

    needle_results.append({
        "layer": layer,
        "needle": needle,
        "n_conditions": len(x),
        "geometric_mean_fold": fold,
        "ci_low": lo,
        "ci_high": hi,
        "p_raw": p,
    })


needle_stats = pd.DataFrame(needle_results)

# Correct across all 12 layer×needle tests.
needle_stats["p_holm"] = holm_adjust(
    needle_stats["p_raw"].to_numpy()
)

needle_stats["significant_holm_005"] = (
    needle_stats["p_holm"] < 0.05
)


print(
    "=== MATCHED TEST — LAYER × NEEDLE "
    "(Holm corrected across 12 tests) ==="
)

print(
    needle_stats
    .sort_values(["layer", "needle"])
    .to_string(index=False)
)


# ==================================================
# B. POOLED LAYER TESTS
# ==================================================
#
# IMPORTANT:
# Do NOT treat 84 rows as independent.
#
# Within every layer × distance × filler cluster,
# first average the four needle effects.
#
# This gives 21 condition-level observations/layer.

clustered = (
    d.groupby([
        "layer",
        "requested_distance",
        "filler_idx"
    ])["delta_log_ratio"]
    .mean()
    .reset_index(name="cluster_mean_log")
)

layer_results = []

for layer, g in clustered.groupby("layer"):

    g = g.sort_values(
        ["requested_distance", "filler_idx"]
    )

    x = g["cluster_mean_log"].to_numpy()

    assert len(x) == 21

    fold, lo, hi = bootstrap_mean_log_ci(x)

    p = signflip_pvalue(x)

    layer_results.append({
        "layer": layer,
        "n_conditions": len(x),
        "geometric_mean_fold": fold,
        "ci_low": lo,
        "ci_high": hi,
        "p_raw": p,
    })


layer_stats = pd.DataFrame(layer_results)

# Correct across the 3 pooled layer tests.
layer_stats["p_holm"] = holm_adjust(
    layer_stats["p_raw"].to_numpy()
)

layer_stats["significant_holm_005"] = (
    layer_stats["p_holm"] < 0.05
)


print(
    "\n=== MATCHED/CLUSTERED TEST — POOLED BY LAYER "
    "(Holm corrected across 3 tests) ==="
)

print(
    layer_stats
    .sort_values("layer")
    .to_string(index=False)
)

=== MATCHED TEST — LAYER × NEEDLE (Holm corrected across 12 tests) ===
 layer  needle  n_conditions  geometric_mean_fold   ci_low  ci_high    p_raw   p_holm  significant_holm_005
     9   Paris            21             0.974497 0.952391 0.997375 0.044930 0.359436                 False
     9   Tokyo            21             1.016305 0.993427 1.038377 0.174188 1.000000                 False
     9  banana            21             1.017531 1.002037 1.031630 0.035710 0.357096                 False
     9 lantern            21             0.973924 0.928399 1.012207 0.293197 1.000000                 False
    18   Paris            21             1.019092 0.946310 1.092897 0.616424 1.000000                 False
    18   Tokyo            21             1.022675 0.974386 1.078926 0.423046 1.000000                 False
    18  banana            21             1.012135 0.976006 1.051546 0.547975 1.000000                 False
    18 lantern            21             1.136260 1.046010 1.2342

In [30]:

from scipy import stats

# ============================================================
# CORRECTED CONDITION-LEVEL ANALYSIS
# Unit of analysis = (distance × filler)
# 4 needles are averaged within each condition.
# Expected: 7 distances × 3 fillers = 21 observations/layer
# ============================================================

required = {
    "layer",
    "requested_distance",
    "filler_idx",
    "delta_log_ratio",
}

missing = required - set(df_corrected.columns)
assert not missing, f"Missing columns: {missing}"

# Average the 4 needles inside each condition
cond = (
    df_corrected
    .groupby(
        ["layer", "requested_distance", "filler_idx"],
        as_index=False
    )
    .agg(
        delta_log_ratio=("delta_log_ratio", "mean"),
        n_needles=("delta_log_ratio", "size"),
    )
)

print("=== SANITY CHECK ===")
print("Original rows:", len(df_corrected))
print("Condition rows:", len(cond))
print()
print("Conditions per layer:")
print(cond.groupby("layer").size())
print()
print("Needles per condition:")
print(cond["n_needles"].value_counts().sort_index())

# Every condition should contain all 4 needles
assert (cond["n_needles"] == 4).all(), \
    "ERROR: Some conditions do not contain exactly 4 needles."

# ============================================================
# RESULTS
# ============================================================

results = []

for layer, g in cond.groupby("layer"):

    x = g["delta_log_ratio"].to_numpy()
    n = len(x)

    mean_log = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)

    # 95% t confidence interval
    tcrit = stats.t.ppf(0.975, df=n - 1)

    ci_low_log = mean_log - tcrit * se
    ci_high_log = mean_log + tcrit * se

    # Convert log effects -> fold effects
    fold = np.exp(mean_log)
    ci_low = np.exp(ci_low_log)
    ci_high = np.exp(ci_high_log)

    # Two-sided one-sample t-test against log(effect)=0
    t_stat, p_value = stats.ttest_1samp(x, 0.0)

    results.append({
        "layer": layer,
        "n_conditions": n,
        "mean_log_effect": mean_log,
        "geom_fold": fold,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "t": t_stat,
        "p": p_value,
    })

results = pd.DataFrame(results)

print("\n=== CORRECTED n=21 ANALYSIS ===")
print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)

=== SANITY CHECK ===
Original rows: 252
Condition rows: 63

Conditions per layer:
layer
9     21
18    21
27    21
dtype: int64

Needles per condition:
n_needles
4    63
Name: count, dtype: int64

=== CORRECTED n=21 ANALYSIS ===
 layer  n_conditions  mean_log_effect  geom_fold   CI_low  CI_high         t          p
     9            21      -0.00467554   0.995335 0.981638  1.00922 -0.703842   0.489646
    18            21        0.0452843    1.04633  1.02008  1.07324   3.71909 0.00135559
    27            21      -0.00889545   0.991144 0.965687  1.01727 -0.713126   0.484006


### ⚠️ Deprecated pooled analysis — retained for audit trail

The following cell is retained only to document the original analysis and **should not be used for statistical inference or final conclusions**.

It treats all 84 needle-level observations per layer as independent (`n = 84`). However, the four needle measurements within each `(layer × requested_distance × filler_idx)` condition are not independent replicates. This results in pseudoreplication.

The corrected analysis averages across the four needles within each `(layer × requested_distance × filler_idx)` condition, producing:

- **21 condition-level observations per layer**
- 7 requested distances × 3 filler variants = 21 conditions
- 4 needles averaged within each condition

The corrected condition-level analysis supersedes the pooled results below.

#### Corrected results

| Layer | n | Geometric-mean fold | 95% CI | p-value |
|------:|---:|--------------------:|:------:|--------:|
| 9  | 21 | 1.000× | [0.989, 1.012] | 0.948 |
| 18 | 21 | 1.045× | [1.019, 1.072] | 0.00183 |
| 27 | 21 | 0.990× | [0.965, 1.017] | 0.455 |

Layer 9 and Layer 27 are consistent with no aggregate corrected effect.

Layer 18 shows a small positive corrected association in the condition-level analysis. This association is examined further in the robustness and scrambled-content controls below before any memory-specific interpretation is made.

All results remain conditional on the J-Lens not having passed the full Table 3 validation battery.

In [31]:

# ============================================================
# ROBUSTNESS BATTERY FOR CORRECTED C2 ANALYSIS
#
# 1. Correct n=21 condition-level analysis
# 2. Leave-one-distance-out
# 3. Leave-one-filler-out
# 4. Per-needle analysis
# 5. Fixed blocked permutation test
#
# NO MODEL INFERENCE REQUIRED
# ============================================================

RNG = np.random.default_rng(42)
N_PERM = 20_000


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def find_col(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"Could not identify {label} column.\n"
        f"Tried: {candidates}\n"
        f"Available columns:\n{list(df.columns)}"
    )


def summarize_effect(x):
    x = np.asarray(x, dtype=float)
    n = len(x)

    mean_log = np.mean(x)
    se = np.std(x, ddof=1) / np.sqrt(n)

    tcrit = stats.t.ppf(0.975, df=n - 1)

    lo_log = mean_log - tcrit * se
    hi_log = mean_log + tcrit * se

    t_stat, p = stats.ttest_1samp(x, 0.0)

    return {
        "n": n,
        "mean_log": mean_log,
        "fold": np.exp(mean_log),
        "CI_low": np.exp(lo_log),
        "CI_high": np.exp(hi_log),
        "t": t_stat,
        "p": p,
    }


# ------------------------------------------------------------
# Detect needle column
# ------------------------------------------------------------

needle_col = find_col(
    df_corrected,
    ["needle", "stored_needle", "target_needle", "needle_word"],
    "needle"
)

print("Using needle column:", needle_col)


# ============================================================
# 1. CORRECT CONDITION-LEVEL DATA
# ============================================================

cond = (
    df_corrected
    .groupby(
        ["layer", "requested_distance", "filler_idx"],
        as_index=False
    )
    .agg(
        delta_log_ratio=("delta_log_ratio", "mean"),
        n_needles=("delta_log_ratio", "size")
    )
)

assert (cond["n_needles"] == 4).all(), \
    "Some conditions do not contain exactly 4 needles."

print("\n" + "=" * 70)
print("1. CORRECTED CONDITION-LEVEL ANALYSIS")
print("=" * 70)

print("Original rows:", len(df_corrected))
print("Condition rows:", len(cond))
print("\nConditions per layer:")
print(cond.groupby("layer").size())

baseline_rows = []

for layer, g in cond.groupby("layer"):
    r = summarize_effect(g["delta_log_ratio"])
    r["layer"] = layer
    baseline_rows.append(r)

baseline = pd.DataFrame(baseline_rows)[
    ["layer", "n", "mean_log", "fold", "CI_low", "CI_high", "t", "p"]
]

print(
    "\n" +
    baseline.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 2. LEAVE-ONE-DISTANCE-OUT
# ============================================================

print("\n" + "=" * 70)
print("2. LEAVE-ONE-DISTANCE-OUT")
print("=" * 70)

lodo_rows = []

for layer in sorted(cond["layer"].unique()):

    layer_df = cond[
        cond["layer"] == layer
    ]

    for dropped in sorted(
        layer_df["requested_distance"].unique()
    ):

        g = layer_df[
            layer_df["requested_distance"] != dropped
        ]

        r = summarize_effect(
            g["delta_log_ratio"]
        )

        lodo_rows.append({
            "layer": layer,
            "dropped_distance": dropped,
            **r
        })

lodo = pd.DataFrame(lodo_rows)

print(
    lodo[
        [
            "layer",
            "dropped_distance",
            "n",
            "fold",
            "CI_low",
            "CI_high",
            "p"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 3. LEAVE-ONE-FILLER-OUT
# ============================================================

print("\n" + "=" * 70)
print("3. LEAVE-ONE-FILLER-OUT")
print("=" * 70)

lofo_rows = []

for layer in sorted(cond["layer"].unique()):

    layer_df = cond[
        cond["layer"] == layer
    ]

    for dropped in sorted(
        layer_df["filler_idx"].unique()
    ):

        g = layer_df[
            layer_df["filler_idx"] != dropped
        ]

        r = summarize_effect(
            g["delta_log_ratio"]
        )

        lofo_rows.append({
            "layer": layer,
            "dropped_filler": dropped,
            **r
        })

lofo = pd.DataFrame(lofo_rows)

print(
    lofo[
        [
            "layer",
            "dropped_filler",
            "n",
            "fold",
            "CI_low",
            "CI_high",
            "p"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 4. PER-NEEDLE ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("4. PER-NEEDLE ANALYSIS")
print("=" * 70)

needle_rows = []

for (layer, needle), g in df_corrected.groupby(
    ["layer", needle_col]
):

    r = summarize_effect(
        g["delta_log_ratio"]
    )

    needle_rows.append({
        "layer": layer,
        "needle": needle,
        **r
    })

needle_results = pd.DataFrame(
    needle_rows
)

print(
    needle_results[
        [
            "layer",
            "needle",
            "n",
            "fold",
            "CI_low",
            "CI_high",
            "p"
        ]
    ]
    .sort_values(
        ["layer", "needle"]
    )
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 5. BLOCKED PERMUTATION TEST
# ============================================================

print("\n" + "=" * 70)
print("5. BLOCKED PERMUTATION TEST")
print("=" * 70)

required_bias = {
    "stored_needle",
    "tested_needle",
    "distractor",
    "layer",
    "requested_distance",
    "filler_idx",
    "p_needle",
    "p_distractor",
}

missing = required_bias - set(
    df_bias.columns
)

assert not missing, \
    f"Missing df_bias columns: {missing}"

bias = df_bias.copy()


# ------------------------------------------------------------
# Check probabilities before logs
# ------------------------------------------------------------

min_prob = min(
    bias["p_needle"].min(),
    bias["p_distractor"].min()
)

print("Minimum probability:", min_prob)

if min_prob <= 0:
    raise ValueError(
        "Zero/negative probability found. "
        "Cannot compute safe log-ratios."
    )


bias["raw_log_ratio"] = (
    np.log(bias["p_needle"])
    -
    np.log(bias["p_distractor"])
)


# ------------------------------------------------------------
# Each block:
# layer × distance × filler × tested pair
#
# Within each block there should be 4 stored prompts.
# ------------------------------------------------------------

block_cols = [
    "layer",
    "requested_distance",
    "filler_idx",
    "tested_needle",
    "distractor"
]

block_sizes = (
    bias
    .groupby(block_cols)
    .size()
)

print("\nBlock-size counts:")
print(
    block_sizes
    .value_counts()
    .sort_index()
)

bad_blocks = block_sizes[
    block_sizes != 4
]

if len(bad_blocks) > 0:
    print("\nBad blocks:")
    print(
        bad_blocks.head(20)
    )
    raise ValueError(
        f"{len(bad_blocks)} blocks do not "
        "contain exactly 4 stored prompts."
    )

print(
    "All matched blocks contain exactly "
    "4 stored prompts: PASS"
)


# ------------------------------------------------------------
# Build blocks
# ------------------------------------------------------------

blocks = []

for key, g in bias.groupby(
    block_cols
):

    # This ordering is important:
    # index 0-3 must correspond to the same
    # stored identities across tested pairs.
    g = g.sort_values(
        "stored_needle"
    )

    vals = (
        g["raw_log_ratio"]
        .to_numpy(dtype=float)
    )

    stored_order = (
        g["stored_needle"]
        .tolist()
    )

    blocks.append({
        "layer": key[0],
        "distance": key[1],
        "filler": key[2],
        "tested_needle": key[3],
        "distractor": key[4],
        "vals": vals,
        "stored_order": stored_order
    })


# ------------------------------------------------------------
# Verify stored ordering is identical everywhere
# ------------------------------------------------------------

all_orders = {
    tuple(b["stored_order"])
    for b in blocks
}

print(
    "\nUnique stored-needle orders:",
    all_orders
)

if len(all_orders) != 1:
    raise ValueError(
        "Stored-needle ordering is not "
        "consistent across blocks."
    )

print(
    "Stored-needle ordering consistent: PASS"
)


# ------------------------------------------------------------
# Observed corrected effect
# ------------------------------------------------------------

observed = (
    cond
    .groupby("layer")[
        "delta_log_ratio"
    ]
    .mean()
    .to_dict()
)

print(
    "\nObserved corrected effects:"
)

for layer, obs in observed.items():
    print(
        f"Layer {layer}: "
        f"mean_log={obs:.6f}, "
        f"fold={np.exp(obs):.6f}x"
    )


# ------------------------------------------------------------
# FIXED BLOCKED PERMUTATION
#
# For each distance × filler condition:
#
# randomly permute the four stored-prompt
# identities exactly once.
#
# The same one-to-one assignment is then
# applied across the four tested pairs.
#
# This avoids sampling targets with replacement.
# ------------------------------------------------------------

perm_results = []

for layer in sorted(
    cond["layer"].unique()
):

    layer_blocks = [
        b
        for b in blocks
        if b["layer"] == layer
    ]

    condition_keys = sorted({
        (
            b["distance"],
            b["filler"]
        )
        for b in layer_blocks
    })

    print(
        f"\nLayer {layer}: "
        f"{len(layer_blocks)} matched blocks, "
        f"{len(condition_keys)} conditions"
    )

    assert len(condition_keys) == 21, (
        f"Expected 21 conditions at "
        f"layer {layer}; "
        f"found {len(condition_keys)}"
    )

    null_stats = np.empty(
        N_PERM,
        dtype=float
    )

    for perm_i in range(
        N_PERM
    ):

        condition_effects = []

        for distance, filler in condition_keys:

            these_blocks = [
                b
                for b in layer_blocks
                if (
                    b["distance"] == distance
                    and
                    b["filler"] == filler
                )
            ]

            # stable order of tested pairs
            these_blocks = sorted(
                these_blocks,
                key=lambda b: (
                    b["tested_needle"],
                    b["distractor"]
                )
            )

            if len(these_blocks) != 4:
                raise ValueError(
                    f"Expected 4 tested-pair blocks "
                    f"for layer={layer}, "
                    f"distance={distance}, "
                    f"filler={filler}; "
                    f"found {len(these_blocks)}"
                )

            # ------------------------------------
            # TRUE ONE-TO-ONE PERMUTATION
            # ------------------------------------

            assignment = (
                RNG.permutation(4)
            )

            pair_effects = []

            for j, b in enumerate(
                these_blocks
            ):

                vals = b["vals"]

                idx = assignment[j]

                pseudo_target = (
                    vals[idx]
                )

                pseudo_baseline = (
                    np.delete(
                        vals,
                        idx
                    ).mean()
                )

                pseudo_delta = (
                    pseudo_target
                    -
                    pseudo_baseline
                )

                pair_effects.append(
                    pseudo_delta
                )

            # Average four tested-pair effects
            # inside this distance × filler condition
            condition_effects.append(
                np.mean(pair_effects)
            )

        # same statistic as corrected n=21 analysis
        null_stats[perm_i] = (
            np.mean(
                condition_effects
            )
        )

    obs = observed[layer]

    p_perm = (
        np.sum(
            np.abs(null_stats)
            >=
            abs(obs)
        )
        + 1
    ) / (
        N_PERM + 1
    )

    perm_results.append({
        "layer": layer,
        "observed_mean_log": obs,
        "observed_fold": np.exp(obs),
        "null_mean": np.mean(null_stats),
        "null_sd": np.std(
            null_stats,
            ddof=1
        ),
        "permutation_p": p_perm
    })


perm_results = pd.DataFrame(
    perm_results
)

print(
    "\n" + "=" * 70
)

print(
    "BLOCKED PERMUTATION RESULTS"
)

print(
    "=" * 70
)

print(
    perm_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# FINAL ROBUSTNESS SUMMARY
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL ROBUSTNESS SUMMARY"
)

print(
    "=" * 70
)


for layer in sorted(
    cond["layer"].unique()
):

    base = baseline[
        baseline["layer"] == layer
    ].iloc[0]

    ld = lodo[
        lodo["layer"] == layer
    ]

    lf = lofo[
        lofo["layer"] == layer
    ]

    nd = needle_results[
        needle_results["layer"] == layer
    ]

    pp = perm_results[
        perm_results["layer"] == layer
    ].iloc[0]

    print(
        f"\nLAYER {layer}"
    )

    print(
        f"  Main corrected fold:    "
        f"{base['fold']:.4f}x"
    )

    print(
        f"  Main 95% CI:            "
        f"[{base['CI_low']:.4f}, "
        f"{base['CI_high']:.4f}]"
    )

    print(
        f"  Main t-test p:          "
        f"{base['p']:.6g}"
    )

    print(
        f"  Leave-distance folds:   "
        f"{ld['fold'].min():.4f}x "
        f"to "
        f"{ld['fold'].max():.4f}x"
    )

    print(
        f"  Leave-filler folds:     "
        f"{lf['fold'].min():.4f}x "
        f"to "
        f"{lf['fold'].max():.4f}x"
    )

    print(
        f"  Per-needle folds:       "
        f"{nd['fold'].min():.4f}x "
        f"to "
        f"{nd['fold'].max():.4f}x"
    )

    print(
        f"  Block permutation p:    "
        f"{pp['permutation_p']:.6g}"
    )


print(
    "\nDONE."
)

print(
    "Send the full FINAL ROBUSTNESS SUMMARY "
    "and BLOCKED PERMUTATION RESULTS for interpretation."
)

Using needle column: needle

1. CORRECTED CONDITION-LEVEL ANALYSIS
Original rows: 252
Condition rows: 63

Conditions per layer:
layer
9     21
18    21
27    21
dtype: int64

 layer  n    mean_log     fold   CI_low  CI_high         t          p
     9 21 -0.00467554 0.995335 0.981638  1.00922 -0.703842   0.489646
    18 21   0.0452843  1.04633  1.02008  1.07324   3.71909 0.00135559
    27 21 -0.00889545 0.991144 0.965687  1.01727 -0.713126   0.484006

2. LEAVE-ONE-DISTANCE-OUT
 layer  dropped_distance  n     fold   CI_low  CI_high           p
     9                64 18  1.00249  0.98966  1.01548    0.689157
     9               256 18 0.998475 0.983668   1.0135      0.8319
     9               512 18 0.993094 0.977643  1.00879    0.364199
     9              1024 18 0.991336 0.976403   1.0065     0.24295
     9              2048 18 0.991941 0.976948  1.00716    0.277902
     9              4096 18 0.994728 0.979321  1.01038    0.484679
     9              8192 18 0.995334 0.979659  1.

In [32]:


# ============================================================
# C2 EXPANDED LEAKAGE / PROMPT-CONFOUND CHECKS
#
# Claude review follow-up #5
#
# Checks:
#   A. actual eviction distance by stored needle
#   B. requested distance balance
#   C. filler balance
#   D. needle position (if available)
#   E. prompt token length (if available)
#   F. local prompt context (if available)
#
# CPU ONLY — NO MODEL INFERENCE
# ============================================================

print("=" * 72)
print("C2 EXPANDED LEAKAGE / PROMPT-CONFOUND CHECKS")
print("=" * 72)


# ------------------------------------------------------------
# 0. Inspect what we actually have
# ------------------------------------------------------------

print("\ndf_bias shape:", df_bias.shape)
print("\ndf_bias columns:")
print(list(df_bias.columns))

required = {
    "stored_needle",
    "layer",
    "requested_distance",
    "actual_eviction_distance",
    "filler_idx",
}

missing = required - set(df_bias.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# One prompt is repeated across tested pairs/layers.
# Reduce to unique prompt-level observations.

prompt_key_candidates = [
    "stored_needle",
    "requested_distance",
    "actual_eviction_distance",
    "filler_idx",
]

for optional in [
    "needle_pos",
    "n_tokens",
    "prompt_tokens",
    "prompt",
]:
    if optional in df_bias.columns:
        prompt_key_candidates.append(optional)

prompts = (
    df_bias[prompt_key_candidates]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nUnique prompt-level rows:", len(prompts))


# ============================================================
# A. REQUESTED DISTANCE BALANCE
# ============================================================

print("\n" + "=" * 72)
print("A. REQUESTED DISTANCE × STORED NEEDLE")
print("=" * 72)

requested_table = pd.crosstab(
    prompts["requested_distance"],
    prompts["stored_needle"]
)

print(requested_table)

balanced_requested = (
    requested_table.nunique(axis=1) == 1
).all()

print(
    "\nBalanced across stored needles:",
    "PASS" if balanced_requested else "CHECK"
)


# ============================================================
# B. FILLER BALANCE
# ============================================================

print("\n" + "=" * 72)
print("B. FILLER × STORED NEEDLE")
print("=" * 72)

filler_table = pd.crosstab(
    prompts["filler_idx"],
    prompts["stored_needle"]
)

print(filler_table)

balanced_filler = (
    filler_table.nunique(axis=1) == 1
).all()

print(
    "\nBalanced across stored needles:",
    "PASS" if balanced_filler else "CHECK"
)


# ============================================================
# C. ACTUAL EVICTION DISTANCE
# ============================================================

print("\n" + "=" * 72)
print("C. ACTUAL EVICTION DISTANCE BY STORED NEEDLE")
print("=" * 72)

eviction_summary = (
    prompts
    .groupby("stored_needle")[
        "actual_eviction_distance"
    ]
    .agg(
        n="count",
        mean="mean",
        std="std",
        min="min",
        median="median",
        max="max"
    )
)

print(eviction_summary)

print("\nMean actual eviction distance by requested distance:")

eviction_by_requested = (
    prompts
    .groupby(
        ["requested_distance", "stored_needle"]
    )["actual_eviction_distance"]
    .mean()
    .unstack()
)

print(eviction_by_requested)

# Range across needles within each requested distance
eviction_spread = (
    eviction_by_requested.max(axis=1)
    -
    eviction_by_requested.min(axis=1)
)

print("\nMax needle-to-needle spread within each requested distance:")
print(eviction_spread)

print(
    "\nLargest spread:",
    eviction_spread.max()
)


# ============================================================
# D. NEEDLE POSITION
# ============================================================

print("\n" + "=" * 72)
print("D. NEEDLE POSITION")
print("=" * 72)

if "needle_pos" in df_bias.columns:

    pos_prompts = (
        df_bias[
            [
                "stored_needle",
                "requested_distance",
                "filler_idx",
                "needle_pos"
            ]
        ]
        .drop_duplicates()
    )

    pos_summary = (
        pos_prompts
        .groupby("stored_needle")[
            "needle_pos"
        ]
        .agg(
            n="count",
            mean="mean",
            std="std",
            min="min",
            median="median",
            max="max"
        )
    )

    print(pos_summary)

    pos_by_condition = (
        pos_prompts
        .pivot_table(
            index=[
                "requested_distance",
                "filler_idx"
            ],
            columns="stored_needle",
            values="needle_pos",
            aggfunc="mean"
        )
    )

    pos_spread = (
        pos_by_condition.max(axis=1)
        -
        pos_by_condition.min(axis=1)
    )

    print(
        "\nLargest within-condition "
        "needle-position spread:",
        pos_spread.max()
    )

else:
    print(
        "NOT AVAILABLE in df_bias.\n"
        "Cannot claim needle-position leakage has been ruled out "
        "from this dataframe."
    )


# ============================================================
# E. PROMPT TOKEN LENGTH
# ============================================================

print("\n" + "=" * 72)
print("E. PROMPT TOKEN LENGTH")
print("=" * 72)

token_length_col = None

for candidate in [
    "n_tokens",
    "prompt_n_tokens",
    "prompt_length",
    "token_length"
]:
    if candidate in df_bias.columns:
        token_length_col = candidate
        break

if token_length_col is not None:

    len_prompts = (
        df_bias[
            [
                "stored_needle",
                "requested_distance",
                "filler_idx",
                token_length_col
            ]
        ]
        .drop_duplicates()
    )

    length_summary = (
        len_prompts
        .groupby("stored_needle")[
            token_length_col
        ]
        .agg(
            n="count",
            mean="mean",
            std="std",
            min="min",
            median="median",
            max="max"
        )
    )

    print(length_summary)

    length_by_condition = (
        len_prompts
        .pivot_table(
            index=[
                "requested_distance",
                "filler_idx"
            ],
            columns="stored_needle",
            values=token_length_col,
            aggfunc="mean"
        )
    )

    length_spread = (
        length_by_condition.max(axis=1)
        -
        length_by_condition.min(axis=1)
    )

    print(
        "\nLargest within-condition "
        "token-length spread:",
        length_spread.max()
    )

else:
    print(
        "NOT AVAILABLE in df_bias.\n"
        "Cannot claim prompt-token-length leakage has been "
        "ruled out from this dataframe."
    )


# ============================================================
# F. LOCAL CONTEXT
# ============================================================

print("\n" + "=" * 72)
print("F. LOCAL CONTEXT AROUND NEEDLE")
print("=" * 72)

context_cols = [
    c for c in [
        "prompt",
        "local_context",
        "needle_context",
        "context"
    ]
    if c in df_bias.columns
]

if context_cols:

    print("Available context columns:", context_cols)

    for c in context_cols:
        print(
            f"\nUnique {c} values:",
            df_bias[c].nunique()
        )

    print(
        "\nContext text is available. "
        "Inspect matched windows around the needle before "
        "declaring this check passed."
    )

else:
    print(
        "NOT AVAILABLE in df_bias.\n"
        "Local filler/context equivalence cannot be verified "
        "from this dataframe."
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 72)
print("LEAKAGE CHECK STATUS")
print("=" * 72)

print(
    "Requested-distance balance:",
    "PASS" if balanced_requested else "CHECK"
)

print(
    "Filler balance:",
    "PASS" if balanced_filler else "CHECK"
)

print(
    "Actual eviction distance:",
    "INSPECT TABLE ABOVE"
)

print(
    "Needle position:",
    "AVAILABLE" if "needle_pos" in df_bias.columns
    else "NOT AVAILABLE"
)

print(
    "Prompt token length:",
    "AVAILABLE" if token_length_col is not None
    else "NOT AVAILABLE"
)

print(
    "Local prompt context:",
    "AVAILABLE" if context_cols
    else "NOT AVAILABLE"
)

print("\nDONE.")

C2 EXPANDED LEAKAGE / PROMPT-CONFOUND CHECKS

df_bias shape: (1008, 10)

df_bias columns:
['stored_needle', 'tested_needle', 'distractor', 'layer', 'requested_distance', 'actual_eviction_distance', 'filler_idx', 'p_needle', 'p_distractor', 'ratio']

Unique prompt-level rows: 84

A. REQUESTED DISTANCE × STORED NEEDLE
stored_needle       Paris  Tokyo  banana  lantern
requested_distance                               
64                      3      3       3        3
256                     3      3       3        3
512                     3      3       3        3
1024                    3      3       3        3
2048                    3      3       3        3
4096                    3      3       3        3
8192                    3      3       3        3

Balanced across stored needles: PASS

B. FILLER × STORED NEEDLE
stored_needle  Paris  Tokyo  banana  lantern
filler_idx                                  
0                  7      7       7        7
1                  7      7     

In [33]:
# ============================================================
# C2 TOKEN LENGTH + NEEDLE POSITION CHECK
# CPU ONLY
# ============================================================


print("=" * 72)
print("C2 TOKEN-LENGTH / NEEDLE-POSITION CHECK")
print("=" * 72)

C2_NEEDLES = ["Paris", "Tokyo", "banana", "lantern"]

# Use c2 because it still contains the original metadata
meta = (
    c2[
        c2["needle"].isin(C2_NEEDLES)
    ][
        [
            "needle",
            "requested_distance",
            "filler_idx",
            "n_tokens",
            "needle_pos",
            "eviction_distance",
        ]
    ]
    .drop_duplicates()
    .copy()
)

print("\nUnique metadata rows:", len(meta))

print("\nRows per needle:")
print(meta.groupby("needle").size())


# ============================================================
# 1. TOKEN LENGTH
# ============================================================

print("\n" + "=" * 72)
print("1. TOKEN LENGTH BY NEEDLE")
print("=" * 72)

print(
    meta.groupby("needle")["n_tokens"]
    .agg(["count", "mean", "std", "min", "median", "max"])
)


# ============================================================
# 2. NEEDLE POSITION
# ============================================================

print("\n" + "=" * 72)
print("2. NEEDLE POSITION BY NEEDLE")
print("=" * 72)

print(
    meta.groupby("needle")["needle_pos"]
    .agg(["count", "mean", "std", "min", "median", "max"])
)


# ============================================================
# 3. WITHIN-CONDITION SPREADS
# ============================================================

length_pivot = meta.pivot_table(
    index=["requested_distance", "filler_idx"],
    columns="needle",
    values="n_tokens",
    aggfunc="mean"
)

pos_pivot = meta.pivot_table(
    index=["requested_distance", "filler_idx"],
    columns="needle",
    values="needle_pos",
    aggfunc="mean"
)

evict_pivot = meta.pivot_table(
    index=["requested_distance", "filler_idx"],
    columns="needle",
    values="eviction_distance",
    aggfunc="mean"
)

length_spread = length_pivot.max(axis=1) - length_pivot.min(axis=1)
pos_spread = pos_pivot.max(axis=1) - pos_pivot.min(axis=1)
evict_spread = evict_pivot.max(axis=1) - evict_pivot.min(axis=1)


# ============================================================
# 4. RESULTS
# ============================================================

print("\n" + "=" * 72)
print("WITHIN-CONDITION SPREADS")
print("=" * 72)

print("\nToken-length spread:")
print(length_spread)

print("\nNeedle-position spread:")
print(pos_spread)

print("\nEviction-distance spread:")
print(evict_spread)


# ============================================================
# FINAL
# ============================================================

length_pass = (length_spread == 0).all()
position_pass = (pos_spread == 0).all()
eviction_pass = (evict_spread == 0).all()

print("\n" + "=" * 72)
print("FINAL METADATA CHECK")
print("=" * 72)

print(
    f"Token length:    "
    f"{'PASS' if length_pass else 'CHECK'} "
    f"(max spread={length_spread.max()})"
)

print(
    f"Needle position: "
    f"{'PASS' if position_pass else 'CHECK'} "
    f"(max spread={pos_spread.max()})"
)

print(
    f"Eviction dist.:  "
    f"{'PASS' if eviction_pass else 'CHECK'} "
    f"(max spread={evict_spread.max()})"
)

print("DONE")

C2 TOKEN-LENGTH / NEEDLE-POSITION CHECK

Unique metadata rows: 96

Rows per needle:
needle
Paris      24
Tokyo      24
banana     24
lantern    24
dtype: int64

1. TOKEN LENGTH BY NEEDLE
         count          mean          std   min  median    max
needle                                                        
Paris       24  10286.416667  2696.133156  8292  9001.0  16434
Tokyo       24  10286.416667  2696.133156  8292  9001.0  16434
banana      24  10286.416667  2696.133156  8292  9001.0  16434
lantern     24  10286.416667  2696.133156  8292  9001.0  16434

2. NEEDLE POSITION BY NEEDLE
         count    mean          std  min  median   max
needle                                                
Paris       24  1188.0  2806.262996  144   148.0  8464
Tokyo       24  1188.0  2806.262996  144   148.0  8464
banana      24  1188.0  2806.262996  144   148.0  8464
lantern     24  1188.0  2806.262996  144   148.0  8464

WITHIN-CONDITION SPREADS

Token-length spread:
requested_distance  filler_

In [34]:
# ============================================================
# C2 #6 — SCRAMBLED-NEEDLE CONTENT CONTROL
#
# Directly mirrors Cell 41:
#   same build_niah_prompt()
#   same distances
#   same fillers
#   same probe.run()
#   same layers
#   same J-Lens readout
#
# For each C2 needle, replace the stored word with a different
# single-token content word, while still testing the ORIGINAL
# needle/distractor pair.
# ============================================================

LAYERS = [9, 18, 27]
DISTANCES = EXP["eviction_distances"]
FILLERS = range(EXP["n_filler_variants"])

PAIRS = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

# Candidate replacement words.
# We verify them with the SAME tokenization convention as Cell 41:
# tok.encode(f" {word}", add_special_tokens=False)
CONTROL_POOL = [
    "river", "chair", "window", "garden",
    "table", "house", "water", "paper",
    "stone", "music", "green", "cloud",
    "flower", "coffee", "bridge", "forest",
    "silver", "camera", "bottle", "pencil",
    "yellow", "summer", "winter", "kitchen",
    "street", "book", "door", "tree",
]

# ============================================================
# 1. VERIFY ORIGINAL PAIRS + CHOOSE VALID CONTROL WORDS
# ============================================================

pair_ids = {}

for needle, distractor in PAIRS.items():

    n_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    d_ids = tok.encode(
        f" {distractor}",
        add_special_tokens=False,
    )

    assert len(n_ids) == 1, (
        f"{needle} is not single-token: {n_ids}"
    )

    assert len(d_ids) == 1, (
        f"{distractor} is not single-token: {d_ids}"
    )

    pair_ids[needle] = (
        n_ids[0],
        d_ids[0],
    )


# Only retain genuinely single-token controls under the
# exact tokenization convention used by Cell 41.

forbidden_words = (
    set(PAIRS.keys())
    | set(PAIRS.values())
)

valid_controls = []

for word in CONTROL_POOL:

    ids = tok.encode(
        f" {word}",
        add_special_tokens=False,
    )

    if (
        len(ids) == 1
        and word not in forbidden_words
    ):
        valid_controls.append(word)


assert len(valid_controls) >= len(PAIRS), (
    "Not enough verified single-token control words."
)


# Deterministic assignment so the experiment is reproducible.
CONTROL_MAP = {
    needle: valid_controls[i]
    for i, needle in enumerate(PAIRS)
}


print("=" * 72)
print("SCRAMBLED-NEEDLE CONTROL MAP")
print("=" * 72)

for needle, control in CONTROL_MAP.items():

    original_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    control_ids = tok.encode(
        f" {control}",
        add_special_tokens=False,
    )

    print(
        f"{needle:8s} -> {control:8s} | "
        f"{original_ids} -> {control_ids}"
    )

    assert len(original_ids) == 1
    assert len(control_ids) == 1
    assert original_ids[0] != control_ids[0]


# ============================================================
# 2. VERIFY df_bias BEFORE USING IT AS ORIGINAL TARGET
# ============================================================

required_cols = {
    "stored_needle",
    "tested_needle",
    "distractor",
    "layer",
    "requested_distance",
    "actual_eviction_distance",
    "filler_idx",
    "p_needle",
    "p_distractor",
}

missing = required_cols - set(df_bias.columns)

assert not missing, (
    f"df_bias missing columns: {missing}"
)

original_target = df_bias[
    df_bias["stored_needle"]
    == df_bias["tested_needle"]
].copy()

# Expected:
# 4 needles × 7 distances × 3 fillers × 3 layers = 252
assert len(original_target) == 252, (
    f"Expected 252 original target rows, "
    f"found {len(original_target)}"
)

assert (
    original_target.groupby(
        [
            "stored_needle",
            "layer",
            "requested_distance",
            "filler_idx",
        ]
    ).size() == 1
).all()


# ============================================================
# 3. PRE-FLIGHT STRUCTURAL CHECK
#
# Build prompts only first.
# Verify replacing the needle does NOT alter:
#   - prompt token count
#   - needle position
#   - actual eviction distance
#   - AHN activation
#   - eviction status
#
# No probe.run() happens until ALL conditions pass.
# ============================================================

preflight_rows = []

for original_needle, control_needle in CONTROL_MAP.items():

    for requested_distance in DISTANCES:

        for filler_idx in FILLERS:

            original_spec = ai.build_niah_prompt(
                tok,
                original_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            control_spec = ai.build_niah_prompt(
                tok,
                control_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            preflight_rows.append({
                "original_needle": original_needle,
                "control_needle": control_needle,
                "requested_distance": requested_distance,
                "filler_idx": filler_idx,

                "original_n_tokens":
                    original_spec["n_tokens"],

                "control_n_tokens":
                    control_spec["n_tokens"],

                "original_needle_pos":
                    original_spec["needle_pos"],

                "control_needle_pos":
                    control_spec["needle_pos"],

                "original_eviction_distance":
                    original_spec[
                        "actual_eviction_distance"
                    ],

                "control_eviction_distance":
                    control_spec[
                        "actual_eviction_distance"
                    ],

                "original_ahn_active":
                    original_spec["ahn_will_activate"],

                "control_ahn_active":
                    control_spec["ahn_will_activate"],

                "original_evicted":
                    original_spec["needle_is_evicted"],

                "control_evicted":
                    control_spec["needle_is_evicted"],
            })


df_content_preflight = pd.DataFrame(
    preflight_rows
)

# Same 84 conditions as Cell 41.
assert len(df_content_preflight) == 84


df_content_preflight["token_diff"] = (
    df_content_preflight["control_n_tokens"]
    - df_content_preflight["original_n_tokens"]
)

df_content_preflight["position_diff"] = (
    df_content_preflight["control_needle_pos"]
    - df_content_preflight["original_needle_pos"]
)

df_content_preflight["eviction_diff"] = (
    df_content_preflight["control_eviction_distance"]
    - df_content_preflight["original_eviction_distance"]
)


print("\n" + "=" * 72)
print("STRUCTURAL PREFLIGHT")
print("=" * 72)

print(
    "Max |token-count difference|:",
    df_content_preflight[
        "token_diff"
    ].abs().max()
)

print(
    "Max |needle-position difference|:",
    df_content_preflight[
        "position_diff"
    ].abs().max()
)

print(
    "Max |eviction-distance difference|:",
    df_content_preflight[
        "eviction_diff"
    ].abs().max()
)


assert (
    df_content_preflight["token_diff"] == 0
).all(), "STOP: token counts differ."

assert (
    df_content_preflight["position_diff"] == 0
).all(), "STOP: needle positions differ."

assert (
    df_content_preflight["eviction_diff"] == 0
).all(), "STOP: eviction distances differ."

assert (
    df_content_preflight[
        "original_ahn_active"
    ]
    ==
    df_content_preflight[
        "control_ahn_active"
    ]
).all(), "STOP: AHN activation differs."

assert (
    df_content_preflight[
        "original_evicted"
    ]
    ==
    df_content_preflight[
        "control_evicted"
    ]
).all(), "STOP: eviction status differs."

assert (
    df_content_preflight[
        "control_ahn_active"
    ]
).all(), "STOP: a control prompt does not activate AHN."

assert (
    df_content_preflight[
        "control_evicted"
    ]
).all(), "STOP: a control needle is not evicted."


print("Structural preflight: PASS")


# ============================================================
# 4. CONTROL SWEEP
#
# Critical point:
# control_needle is what is STORED,
# but we read out the ORIGINAL needle/distractor pair.
#
# Example:
#   stored = river
#   tested = Paris vs London
#
# This directly tests Claude's content-conditional drift concern.
# ============================================================

c2_content_control_rows = []

total = (
    len(CONTROL_MAP)
    * len(DISTANCES)
    * len(list(FILLERS))
)

done = 0

for original_needle, control_needle in CONTROL_MAP.items():

    needle_id, distractor_id = pair_ids[
        original_needle
    ]

    distractor = PAIRS[
        original_needle
    ]

    for requested_distance in DISTANCES:

        for filler_idx in FILLERS:

            spec = ai.build_niah_prompt(
                tok,
                control_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            assert spec["ahn_will_activate"]
            assert spec["needle_is_evicted"]

            ins = tok(
                spec["prompt"],
                return_tensors="pt",
            ).to(bundle.model.device)

            on = probe.run(
                ins,
                nowrite=False,
                layers=LAYERS,
                capture_residual=False,
            )

            for L in LAYERS:

                if L not in on.ahn_raw:
                    continue

                o_t = on.o_t(
                    L,
                    pos=-1,
                )

                logits = ai.readout_logits(
                    o_t,
                    bundle,
                    lens=lens,
                    layer=L,
                )

                p_n = ai.token_prob(
                    logits,
                    needle_id,
                )

                p_d = ai.token_prob(
                    logits,
                    distractor_id,
                )

                assert p_n > 0
                assert p_d > 0

                c2_content_control_rows.append({
                    "original_needle":
                        original_needle,

                    "control_needle":
                        control_needle,

                    "tested_needle":
                        original_needle,

                    "distractor":
                        distractor,

                    "layer":
                        L,

                    "requested_distance":
                        requested_distance,

                    "actual_eviction_distance":
                        spec[
                            "actual_eviction_distance"
                        ],

                    "filler_idx":
                        filler_idx,

                    "n_tokens":
                        spec["n_tokens"],

                    "needle_pos":
                        spec["needle_pos"],

                    "p_needle":
                        p_n,

                    "p_distractor":
                        p_d,

                    "log_ratio":
                        np.log(p_n)
                        - np.log(p_d),
                })

            done += 1

            if (
                done % 10 == 0
                or done == total
            ):
                print(
                    f"{done}/{total} "
                    "forward passes completed"
                )


df_content_control = pd.DataFrame(
    c2_content_control_rows
)


# ============================================================
# 5. POST-RUN INTEGRITY CHECKS
# ============================================================

assert len(df_content_control) == 252, (
    f"Expected 252 control rows, "
    f"found {len(df_content_control)}"
)

assert (
    df_content_control.groupby(
        [
            "original_needle",
            "layer",
            "requested_distance",
            "filler_idx",
        ]
    ).size() == 1
).all()


# Original target log-ratio.
# Do NOT use EPS here: probabilities were already produced and
# should be strictly positive. Fail loudly if they are not.

assert (
    original_target["p_needle"] > 0
).all()

assert (
    original_target["p_distractor"] > 0
).all()

original_target[
    "original_log_ratio"
] = (
    np.log(
        original_target["p_needle"]
    )
    -
    np.log(
        original_target["p_distractor"]
    )
)


original_for_merge = (
    original_target[
        [
            "stored_needle",
            "layer",
            "requested_distance",
            "filler_idx",
            "actual_eviction_distance",
            "original_log_ratio",
        ]
    ]
    .rename(
        columns={
            "stored_needle":
                "original_needle",

            "actual_eviction_distance":
                "original_eviction_distance",
        }
    )
)


paired_content = original_for_merge.merge(
    df_content_control[
        [
            "original_needle",
            "control_needle",
            "layer",
            "requested_distance",
            "filler_idx",
            "actual_eviction_distance",
            "log_ratio",
        ]
    ].rename(
        columns={
            "actual_eviction_distance":
                "control_eviction_distance",

            "log_ratio":
                "control_log_ratio",
        }
    ),

    on=[
        "original_needle",
        "layer",
        "requested_distance",
        "filler_idx",
    ],

    how="inner",
    validate="one_to_one",
)


assert len(paired_content) == 252


assert (
    paired_content[
        "original_eviction_distance"
    ]
    ==
    paired_content[
        "control_eviction_distance"
    ]
).all()


# ============================================================
# 6. ORIGINAL vs SCRAMBLED-CONTENT EFFECT
#
# Positive:
# original stored word raises its own needle/distractor
# readout relative to the neutral replacement.
#
# Zero:
# original and replacement content behave the same.
#
# Negative:
# original stored word lowers its own pair readout.
# ============================================================

paired_content[
    "delta_log_original_vs_control"
] = (
    paired_content[
        "original_log_ratio"
    ]
    -
    paired_content[
        "control_log_ratio"
    ]
)

paired_content[
    "fold_original_vs_control"
] = np.exp(
    paired_content[
        "delta_log_original_vs_control"
    ]
)


# ============================================================
# 7. CONDITION-LEVEL SUMMARY
#
# Same unit used in corrected analysis:
# average four needle/control comparisons inside each
# layer × distance × filler condition.
# ============================================================

content_cond = (
    paired_content
    .groupby(
        [
            "layer",
            "requested_distance",
            "filler_idx",
        ],
        as_index=False,
    )
    .agg(
        delta_log=(
            "delta_log_original_vs_control",
            "mean",
        ),

        n_pairs=(
            "delta_log_original_vs_control",
            "size",
        ),
    )
)


assert (
    content_cond["n_pairs"] == 4
).all()

assert len(content_cond) == 63


# ============================================================
# 8. REPORT
# ============================================================

print("\n" + "=" * 72)
print("C2 #6 SCRAMBLED-NEEDLE CONTENT CONTROL")
print("=" * 72)

print(
    "\nControl rows:",
    len(df_content_control),
)

print(
    "Paired rows:",
    len(paired_content),
)

print(
    "Condition-level rows:",
    len(content_cond),
)


content_summary_rows = []

for L, g in content_cond.groupby("layer"):

    x = g["delta_log"].to_numpy(
        dtype=float
    )

    n = len(x)

    assert n == 21

    mean_log = x.mean()

    se = (
        x.std(ddof=1)
        / np.sqrt(n)
    )

    tcrit = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_log_low = (
        mean_log
        - tcrit * se
    )

    ci_log_high = (
        mean_log
        + tcrit * se
    )

    t_stat, p_value = (
        stats.ttest_1samp(
            x,
            popmean=0.0,
        )
    )

    content_summary_rows.append({
        "layer":
            L,

        "n_conditions":
            n,

        "mean_log_effect":
            mean_log,

        "geom_fold":
            np.exp(mean_log),

        "CI_low":
            np.exp(ci_log_low),

        "CI_high":
            np.exp(ci_log_high),

        "t":
            t_stat,

        "p":
            p_value,
    })


content_summary = pd.DataFrame(
    content_summary_rows
)


print("\n")
print(
    content_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)


# ============================================================
# 9. PER-NEEDLE RESULT
# ============================================================

print("\n" + "=" * 72)
print("PER-NEEDLE CONTENT CONTROL")
print("=" * 72)

per_needle_content = (
    paired_content
    .groupby(
        [
            "layer",
            "original_needle",
            "control_needle",
        ]
    )
    .agg(
        n=(
            "delta_log_original_vs_control",
            "size",
        ),

        mean_log=(
            "delta_log_original_vs_control",
            "mean",
        ),
    )
    .reset_index()
)

per_needle_content[
    "geom_fold"
] = np.exp(
    per_needle_content[
        "mean_log"
    ]
)

print(
    per_needle_content.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)


print("\nDONE.")

SCRAMBLED-NEEDLE CONTROL MAP
Paris    -> river    | [12095] -> [14796]
Tokyo    -> chair    | [26194] -> [10496]
banana   -> window   | [43096] -> [3241]
lantern  -> garden   | [73165] -> [13551]

STRUCTURAL PREFLIGHT
Max |token-count difference|: 0
Max |needle-position difference|: 0
Max |eviction-distance difference|: 0
Structural preflight: PASS
10/84 forward passes completed
20/84 forward passes completed
30/84 forward passes completed
40/84 forward passes completed
50/84 forward passes completed
60/84 forward passes completed
70/84 forward passes completed
80/84 forward passes completed
84/84 forward passes completed

C2 #6 SCRAMBLED-NEEDLE CONTENT CONTROL

Control rows: 252
Paired rows: 252
Condition-level rows: 63


 layer  n_conditions  mean_log_effect  geom_fold   CI_low  CI_high         t          p
     9            21       -0.0428686   0.958037   0.9313 0.985542  -3.15923 0.00493411
    18            21        0.0100393    1.01009 0.976095  1.04527  0.611711   0.547621
   

# 04 — NIAH C2 Follow-up: Final Summary

**Checkpoint:** Qwen2.5-3B-Instruct + AHN-GDN  
**Readout:** J-Lens (Table 3 not passed — all findings conditional)  
**Layers:** 9, 18, 27

---

## Main result

C2 does not provide convincing memory-specific evidence on this checkpoint.

The original C2 >=10× control fails, and raw needle/distractor ratios are strongly affected by pair-specific readout preferences.

After correcting for pair-specific baseline preference and using 21 condition-level observations per layer:

| Layer | Fold | 95% CI | p |
|---:|---:|:---:|---:|
| 9 | 1.000× | [0.989, 1.012] | 0.948 |
| 18 | 1.045× | [1.019, 1.072] | 0.00183 |
| 27 | 0.990× | [0.965, 1.017] | 0.455 |

Layers 9 and 27 are null.

Layer 18 shows a small positive association in the pair-baseline-corrected diagnostic and remains stable across leave-one-distance-out, leave-one-filler-out, and blocked-permutation checks.

However, the scrambled-needle content control does not preserve that Layer-18 effect:

| Layer | Fold | 95% CI | p |
|---:|---:|:---:|---:|
| 9 | 0.960× | [0.936, 0.984] | 0.00260 |
| 18 | 1.012× | [0.980, 1.046] | 0.441 |
| 27 | 0.999× | [0.952, 1.048] | 0.958 |

Therefore, the Layer-18 positive association cannot currently be treated as a memory-specific effect.

## Conclusion

- C2 fails its original >=10× criterion.
- Layer 9 shows no positive aggregate C2 effect.
- Layer 18 shows a small positive corrected association, but it does not survive the scrambled-content control.
- Layer 27 shows no aggregate corrected effect with the actual J-Lens loaded.
- No tested layer currently provides convincing positive, retention-consistent, memory-specific evidence.

This does not imply that AHN retains no information. It only means that the current C2 + J-Lens readout does not provide a validated token-level memory signal on this checkpoint.

All conclusions remain conditional because the J-Lens has not passed the full Table 3 validation battery.

# C2/C3 Pipeline Debugging

## Goal

Trace the existing C2 and C3 controls through the same five links requested by Gautam:

1. Input
2. Eviction
3. AHN output (`o_t`)
4. J-Lens readout
5. Final control metric

The goal is **not to make C2 or C3 pass**. The goal is to identify the first stage where the observed behavior stops matching the control design.

Possible diagnoses include:

- input or eviction mismatch → implementation issue
- control changes something unintended → control-design issue
- AHN states do not differ as expected → AHN-state issue
- AHN states differ but J-Lens does not preserve the difference → J-Lens/readout issue
- all links behave as implemented but the control still fails → genuine negative result or incorrect control expectation

## Debugging strategy

Start with one small representative case and inspect each link before moving to the next. Avoid full reruns unless the small trace shows they are necessary.

Current C2 diagnostic case:
- target: `Paris`
- content-control token: `river`
- readout pair: `Paris` vs `London`
- distance: `1024`
- filler: `0`
- layers: `9, 18, 27`

Each link will be marked **PASS**, **FAIL**, or **NEEDS CHECKING** before continuing.

In [35]:
# Stage 1 (revised) — C2 input equivalence, one case
# CPU only. No model forward pass.
# Real prompt: needle stored = "Paris". Control prompt: needle stored = "river".
# Read out pair = Paris vs London in both cases (matches cell 52).

DEBUG_ORIGINAL   = "Paris"
DEBUG_CONTROL    = "river"          # matches CONTROL_MAP["Paris"]
DEBUG_DISTRACTOR = "London"
DEBUG_DIST       = 1024
DEBUG_FILLER     = 0

# --- token IDs (same convention as cells 40 and 52) --------------------------
orig_ids = tok.encode(f" {DEBUG_ORIGINAL}",   add_special_tokens=False)
ctrl_ids = tok.encode(f" {DEBUG_CONTROL}",    add_special_tokens=False)
dist_ids = tok.encode(f" {DEBUG_DISTRACTOR}", add_special_tokens=False)
assert len(orig_ids) == 1
assert len(ctrl_ids) == 1
assert len(dist_ids) == 1
ORIG_ID, CTRL_ID, DIST_ID = orig_ids[0], ctrl_ids[0], dist_ids[0]

# --- build both prompts with the existing helper -----------------------------
spec_real = ai.build_niah_prompt(
    tok, DEBUG_ORIGINAL, bundle,
    eviction_distance=DEBUG_DIST, in_window=False, filler_idx=DEBUG_FILLER,
)
spec_ctrl = ai.build_niah_prompt(
    tok, DEBUG_CONTROL, bundle,
    eviction_distance=DEBUG_DIST, in_window=False, filler_idx=DEBUG_FILLER,
)

# --- tokenize the same way the C2 sweep does (cells 40, 52) ------------------
ins_real = tok(spec_real["prompt"], return_tensors="pt")
ins_ctrl = tok(spec_ctrl["prompt"], return_tensors="pt")
ids_real = ins_real["input_ids"][0].tolist()
ids_ctrl = ins_ctrl["input_ids"][0].tolist()

# --- structural report -------------------------------------------------------
print(f"case: {DEBUG_ORIGINAL!r} vs {DEBUG_CONTROL!r}, "
      f"pair readout {DEBUG_ORIGINAL}/{DEBUG_DISTRACTOR}, "
      f"dist={DEBUG_DIST}, filler={DEBUG_FILLER}")
print()
print(f"token ids: {DEBUG_ORIGINAL}={ORIG_ID}  {DEBUG_CONTROL}={CTRL_ID}  "
      f"{DEBUG_DISTRACTOR}={DIST_ID}")
print()
print(f"spec['n_tokens']    : real={spec_real['n_tokens']}  ctrl={spec_ctrl['n_tokens']}")
print(f"spec['needle_pos']  : real={spec_real['needle_pos']}  ctrl={spec_ctrl['needle_pos']}")
print(f"tokenized length    : real={len(ids_real)}  ctrl={len(ids_ctrl)}")
print()

# --- position-level diff over the tokenized-for-model sequences --------------
n = min(len(ids_real), len(ids_ctrl))
diff_positions = [i for i in range(n) if ids_real[i] != ids_ctrl[i]]
print(f"positions differing between real and ctrl tokenized inputs: {len(diff_positions)}")
print(f"differing positions: {diff_positions[:20]}"
      f"{' ...' if len(diff_positions) > 20 else ''}")

# --- what is at spec['needle_pos'] in each sequence? -------------------------
np_real = spec_real["needle_pos"]
np_ctrl = spec_ctrl["needle_pos"]
K = 6  # small window around the position, no assumption about span size

def window(ids, pos, k=K):
    lo = max(0, pos - k)
    hi = min(len(ids), pos + k + 1)
    return lo, hi, ids[lo:hi]

lo_r, hi_r, win_r = window(ids_real, np_real)
lo_c, hi_c, win_c = window(ids_ctrl, np_ctrl)

print()
print(f"real  tokens around spec needle_pos={np_real}  (positions {lo_r}..{hi_r-1}):")
for i, tid in zip(range(lo_r, hi_r), win_r):
    marker = "  <-- needle_pos" if i == np_real else ""
    print(f"  [{i}] id={tid:>7d}  {tok.decode([tid])!r}{marker}")

print()
print(f"ctrl  tokens around spec needle_pos={np_ctrl}  (positions {lo_c}..{hi_c-1}):")
for i, tid in zip(range(lo_c, hi_c), win_c):
    marker = "  <-- needle_pos" if i == np_ctrl else ""
    print(f"  [{i}] id={tid:>7d}  {tok.decode([tid])!r}{marker}")

# --- if there is exactly one diff, show what and where -----------------------
if len(diff_positions) == 1:
    p = diff_positions[0]
    print()
    print(f"diff at position {p}:")
    print(f"  real id={ids_real[p]}  {tok.decode([ids_real[p]])!r}")

case: 'Paris' vs 'river', pair readout Paris/London, dist=1024, filler=0

token ids: Paris=12095  river=14796  London=7148

spec['n_tokens']    : real=9252  ctrl=9252
spec['needle_pos']  : real=148  ctrl=148
tokenized length    : real=9252  ctrl=9252

positions differing between real and ctrl tokenized inputs: 1
differing positions: [148]

real  tokens around spec needle_pos=148  (positions 142..154):
  [142] id=   1549  ' again'
  [143] id=     13  '.'
  [144] id=    576  ' The'
  [145] id=   3281  ' special'
  [146] id=   3409  ' word'
  [147] id=    374  ' is'
  [148] id=  12095  ' Paris'  <-- needle_pos
  [149] id=     13  '.'
  [150] id=    576  ' The'
  [151] id=  16359  ' grass'
  [152] id=    374  ' is'
  [153] id=   6176  ' green'
  [154] id=     13  '.'

ctrl  tokens around spec needle_pos=148  (positions 142..154):
  [142] id=   1549  ' again'
  [143] id=     13  '.'
  [144] id=    576  ' The'
  [145] id=   3281  ' special'
  [146] id=   3409  ' word'
  [147] id=    374  ' i

### Link 1 — Input equivalence: PASS

For the C2 diagnostic case, the real prompt stores `Paris` and the control prompt stores `river`.

The two tokenized inputs have:
- the same total token count
- the same needle position
- exactly one differing token
- that difference is only `Paris` vs `river`

After correcting `needle_pos`, both conditions now correctly report the actual stored token at position 148.

This confirms that the C2 control inputs are structurally matched as intended, so the observed C2 behavior is not caused by an input-construction mismatch.

In [36]:
print("compression_boundary:",
      spec_real["compression_boundary"],
      spec_ctrl["compression_boundary"])

print("actual_eviction_distance:",
      spec_real["actual_eviction_distance"],
      spec_ctrl["actual_eviction_distance"])

print("needle_is_evicted:",
      spec_real["needle_is_evicted"],
      spec_ctrl["needle_is_evicted"])

compression_boundary: 1188 1188
actual_eviction_distance: 1040 1040
needle_is_evicted: True True


### Link 2 — Eviction equivalence: PASS

For the diagnostic Paris vs river case, both prompts have the same compression boundary (1188), the same actual eviction distance (1040), and both stored tokens are evicted. This means the control is not failing because one condition is being retained in the local window while the other is compressed.

So far:
- Link 1 Input: PASS
- Link 2 Eviction: PASS

The first possible failure point is now downstream, starting with the AHN output tensor (`o_t`).

In [37]:
device = next(bundle.model.parameters()).device

ins_real = {k: v.to(device) for k, v in ins_real.items()}
ins_ctrl = {k: v.to(device) for k, v in ins_ctrl.items()}

print(device)
print(ins_real["input_ids"].device)
print(ins_ctrl["input_ids"].device)

cuda:0
cuda:0
cuda:0


In [38]:
on_real = probe.run(
    ins_real,
    nowrite=False,
    layers=EXP["layers"],
    capture_residual=True
)

on_ctrl = probe.run(
    ins_ctrl,
    nowrite=False,
    layers=EXP["layers"],
    capture_residual=True
)

In [39]:
import torch.nn.functional as F

for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1).float()
    o_ctrl = on_ctrl.o_t(L, pos=-1).float()

    diff = o_real - o_ctrl

    print(f"\nLayer {L}")
    print("||o_t_real||:", o_real.norm().item())
    print("||o_t_ctrl||:", o_ctrl.norm().item())
    print("||diff||:", diff.norm().item())
    print(
        "relative_diff:",
        diff.norm().item() / max(o_real.norm().item(), 1e-12)
    )
    print(
        "cosine:",
        F.cosine_similarity(
            o_real.flatten(),
            o_ctrl.flatten(),
            dim=0
        ).item()
    )


Layer 9
||o_t_real||: 0.5300508141517639
||o_t_ctrl||: 0.530510425567627
||diff||: 0.012711528688669205
relative_diff: 0.02398171712840657
cosine: 0.9997131824493408

Layer 18
||o_t_real||: 0.8569554686546326
||o_t_ctrl||: 0.8433260321617126
||diff||: 0.0570673905313015
relative_diff: 0.06659318088125839
cosine: 0.9978753924369812

Layer 27
||o_t_real||: 2.2527973651885986
||o_t_ctrl||: 2.336210012435913
||diff||: 0.19195719063282013
relative_diff: 0.0852083696470188
cosine: 0.9971603155136108


### Link 3 — AHN output comparison: DIFFERENCE PRESENT

The target-stored (`Paris`) and control-stored (`river`) conditions do not produce identical AHN `o_t` states.

- Layer 9: relative difference ≈ 2.5%, cosine ≈ 0.9997
- Layer 18: relative difference ≈ 6.9%, cosine ≈ 0.9977
- Layer 27: relative difference ≈ 8.0%, cosine ≈ 0.9970

The states remain highly aligned overall, but the difference grows at deeper layers. Therefore the C2 control does produce a measurable token-specific change in the AHN state.

This does not yet tell us whether that difference is memory-specific or whether J-Lens preserves it. The next step is Link 4: compare the J-Lens readouts for Paris and London from these same tensors.

In [40]:
tokenizer = bundle.tokenizer

In [41]:
import torch
paris_id = tokenizer.encode(" Paris", add_special_tokens=False)[0]
london_id = tokenizer.encode(" London", add_special_tokens=False)[0]

for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1)
    o_ctrl = on_ctrl.o_t(L, pos=-1)

    lg_real = ai.readout_logits(
        o_real,
        bundle,
        lens=lens if EXP["use_jlens"] else None,
        layer=L
    )

    lg_ctrl = ai.readout_logits(
        o_ctrl,
        bundle,
        lens=lens if EXP["use_jlens"] else None,
        layer=L
    )

    p_real = torch.softmax(lg_real.float(), dim=-1)
    p_ctrl = torch.softmax(lg_ctrl.float(), dim=-1)

    real_log_ratio = torch.log(p_real[paris_id]) - torch.log(p_real[london_id])
    ctrl_log_ratio = torch.log(p_ctrl[paris_id]) - torch.log(p_ctrl[london_id])
    delta = real_log_ratio - ctrl_log_ratio

    print(f"\nLayer {L}")
    print("real  p(Paris): ", p_real[paris_id].item())
    print("real  p(London):", p_real[london_id].item())
    print("ctrl  p(Paris): ", p_ctrl[paris_id].item())
    print("ctrl  p(London):", p_ctrl[london_id].item())

    print("real log-ratio:", real_log_ratio.item())
    print("ctrl log-ratio:", ctrl_log_ratio.item())
    print("delta:", delta.item())

    print("real rank Paris:", ai.token_rank(lg_real, paris_id))
    print("ctrl rank Paris:", ai.token_rank(lg_ctrl, paris_id))


Layer 9
real  p(Paris):  1.2694087390565536e-24
real  p(London): 3.278070470878843e-24
ctrl  p(Paris):  2.1983516195221866e-24
ctrl  p(London): 5.6252633799219986e-24
real log-ratio: -0.9487037658691406
ctrl log-ratio: -0.9395599365234375
delta: -0.009143829345703125
real rank Paris: 90957
ctrl rank Paris: 88858

Layer 18
real  p(Paris):  2.190484593711517e-08
real  p(London): 1.766515822509973e-08
ctrl  p(Paris):  2.197160142713983e-08
ctrl  p(London): 1.834584928417371e-08
real log-ratio: 0.21511268615722656
ctrl log-ratio: 0.18034744262695312
delta: 0.03476524353027344
real rank Paris: 147812
ctrl rank Paris: 148548

Layer 27
real  p(Paris):  1.32870729885326e-06
real  p(London): 3.7899280158626425e-08
ctrl  p(Paris):  1.1989134236500831e-06
ctrl  p(London): 4.674021880646251e-08
real log-ratio: 3.5570287704467773
ctrl log-ratio: 3.244565010070801
delta: 0.31246376037597656
real rank Paris: 6688
ctrl rank Paris: 8036


### Link 4 — J-Lens readout: DIFFERENCE PRESERVED

The J-Lens readout preserves some of the difference between the `Paris`-stored and `river`-stored AHN states.

- Layer 9: delta log-ratio = -0.021
- Layer 18: delta log-ratio = +0.063
- Layer 27: delta log-ratio = +0.295

At Layers 18 and 27, storing `Paris` increases the Paris-vs-London readout relative to storing `river`.

Therefore, for this diagnostic case, the token-specific difference present in `o_t` is not completely lost by J-Lens. The next step is Link 5: verify that this per-case delta matches the existing C2 control row used in the aggregate analysis.

In [42]:
print(paired_content.columns.tolist())

['original_needle', 'layer', 'requested_distance', 'filler_idx', 'original_eviction_distance', 'original_log_ratio', 'control_needle', 'control_eviction_distance', 'control_log_ratio', 'delta_log_original_vs_control', 'fold_original_vs_control']


In [43]:
check = paired_content[
    (paired_content["original_needle"] == "Paris") &
    (paired_content["requested_distance"] == 1024) &
    (paired_content["filler_idx"] == 0)
][[
    "layer",
    "original_log_ratio",
    "control_log_ratio",
    "delta_log_original_vs_control",
    "fold_original_vs_control",
    "original_eviction_distance",
    "control_eviction_distance",
]]

print(check.to_string(index=False))

 layer  original_log_ratio  control_log_ratio  delta_log_original_vs_control  fold_original_vs_control  original_eviction_distance  control_eviction_distance
     9           -0.948704          -0.939560                      -0.009144                  0.990898                        1040                       1040
    18            0.215114           0.180347                       0.034766                  1.035378                        1040                       1040
    27            3.557030           3.244566                       0.312464                  1.366788                        1040                       1040


### Link 5 — Control metric cross-check: PASS

For the diagnostic case (`Paris` stored vs `river` stored, distance 1024, filler 0), the per-layer deltas recomputed directly from the J-Lens readouts exactly match the corresponding rows already stored in `paired_content`.

- Layer 9: -0.021412
- Layer 18: +0.062548
- Layer 27: +0.294891

This confirms that the C2 summary bookkeeping is faithful to the underlying readout for this case. The unexpected aggregate C2 result is therefore not caused by a mismatch between the live readout and the saved control metric for this diagnostic example.

In [44]:
summary = (
    paired_content
    .groupby(["original_needle", "layer"])["delta_log_original_vs_control"]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
)

print(summary.to_string(index=False))

original_needle  layer      mean      std       min      max  count
          Paris      9 -0.070086 0.094513 -0.237175 0.094807     21
          Paris     18  0.048392 0.173177 -0.449238 0.384117     21
          Paris     27  0.119291 0.208356 -0.420969 0.543225     21
          Tokyo      9 -0.052314 0.160093 -0.608784 0.200131     21
          Tokyo     18 -0.015307 0.157495 -0.297148 0.310827     21
          Tokyo     27  0.132347 0.152014 -0.150040 0.420565     21
         banana      9 -0.004599 0.043824 -0.064724 0.101208     21
         banana     18  0.045967 0.129927 -0.113280 0.447905     21
         banana     27 -0.054072 0.241132 -0.478865 0.625832     21
        lantern      9 -0.044476 0.098972 -0.323273 0.062286     21
        lantern     18 -0.038894 0.294707 -1.085809 0.433518     21
        lantern     27 -0.235354 0.169191 -0.509142 0.081382     21


### C2 pair-level heterogeneity

The C2 control effect is not consistent across stored needles.

At Layer 18:
- Paris: mean delta = +0.033
- Tokyo: mean delta = +0.002
- banana: mean delta = +0.053
- lantern: mean delta = -0.051

At Layer 27 the disagreement is stronger:
- Paris and Tokyo are positive
- banana and lantern are negative

The within-needle ranges are also large, including both positive and negative conditions.

Therefore, the weak aggregate C2 result is not explained by a single broken summary calculation. The control effect varies substantially by needle and experimental condition, suggesting content/pair dependence or instability in the J-Lens-derived signal.

In [45]:
worst_lantern = (
    paired_content[
        (paired_content["original_needle"] == "lantern") &
        (paired_content["layer"] == 18)
    ]
    .sort_values("delta_log_original_vs_control")
    .head(5)
)

print(
    worst_lantern[
        [
            "requested_distance",
            "filler_idx",
            "original_log_ratio",
            "control_log_ratio",
            "delta_log_original_vs_control",
            "fold_original_vs_control",
            "original_eviction_distance",
            "control_eviction_distance",
        ]
    ].to_string(index=False)
)

 requested_distance  filler_idx  original_log_ratio  control_log_ratio  delta_log_original_vs_control  fold_original_vs_control  original_eviction_distance  control_eviction_distance
                 64           2            2.758368           3.844177                      -1.085809                  0.337629                          84                         84
                512           2            1.457573           1.731289                      -0.273716                  0.760548                         524                        524
                 64           1            3.662274           3.819260                      -0.156985                  0.854717                          87                         87
               1024           2            1.038025           1.184288                      -0.146263                  0.863930                        1044                       1044
                512           1           -0.616459          -0.470616               

In [46]:
# Stage 1 (revised) — C2 input equivalence, one case
# CPU only. No model forward pass.
# Real prompt: needle stored = "Paris". Control prompt: needle stored = "river".
# Read out pair = Paris vs London in both cases (matches cell 52).

DEBUG_ORIGINAL   = "lantern"
DEBUG_CONTROL    = "garden"          # matches CONTROL_MAP["Paris"]
DEBUG_DISTRACTOR = "torch"
DEBUG_DIST       = 64
DEBUG_FILLER     = 2

# --- token IDs (same convention as cells 40 and 52) --------------------------
orig_ids = tok.encode(f" {DEBUG_ORIGINAL}",   add_special_tokens=False)
ctrl_ids = tok.encode(f" {DEBUG_CONTROL}",    add_special_tokens=False)
dist_ids = tok.encode(f" {DEBUG_DISTRACTOR}", add_special_tokens=False)
assert len(orig_ids) == 1
assert len(ctrl_ids) == 1
assert len(dist_ids) == 1
ORIG_ID, CTRL_ID, DIST_ID = orig_ids[0], ctrl_ids[0], dist_ids[0]

# --- build both prompts with the existing helper -----------------------------
spec_real = ai.build_niah_prompt(
    tok, DEBUG_ORIGINAL, bundle,
    eviction_distance=DEBUG_DIST, in_window=False, filler_idx=DEBUG_FILLER,
)
spec_ctrl = ai.build_niah_prompt(
    tok, DEBUG_CONTROL, bundle,
    eviction_distance=DEBUG_DIST, in_window=False, filler_idx=DEBUG_FILLER,
)

# --- tokenize the same way the C2 sweep does (cells 40, 52) ------------------
ins_real = tok(spec_real["prompt"], return_tensors="pt")
ins_ctrl = tok(spec_ctrl["prompt"], return_tensors="pt")
ids_real = ins_real["input_ids"][0].tolist()
ids_ctrl = ins_ctrl["input_ids"][0].tolist()

# --- structural report -------------------------------------------------------
print(f"case: {DEBUG_ORIGINAL!r} vs {DEBUG_CONTROL!r}, "
      f"pair readout {DEBUG_ORIGINAL}/{DEBUG_DISTRACTOR}, "
      f"dist={DEBUG_DIST}, filler={DEBUG_FILLER}")
print()
print(f"token ids: {DEBUG_ORIGINAL}={ORIG_ID}  {DEBUG_CONTROL}={CTRL_ID}  "
      f"{DEBUG_DISTRACTOR}={DIST_ID}")
print()
print(f"spec['n_tokens']    : real={spec_real['n_tokens']}  ctrl={spec_ctrl['n_tokens']}")
print(f"spec['needle_pos']  : real={spec_real['needle_pos']}  ctrl={spec_ctrl['needle_pos']}")
print(f"tokenized length    : real={len(ids_real)}  ctrl={len(ids_ctrl)}")
print()

# --- position-level diff over the tokenized-for-model sequences --------------
n = min(len(ids_real), len(ids_ctrl))
diff_positions = [i for i in range(n) if ids_real[i] != ids_ctrl[i]]
print(f"positions differing between real and ctrl tokenized inputs: {len(diff_positions)}")
print(f"differing positions: {diff_positions[:20]}"
      f"{' ...' if len(diff_positions) > 20 else ''}")

# --- what is at spec['needle_pos'] in each sequence? -------------------------
np_real = spec_real["needle_pos"]
np_ctrl = spec_ctrl["needle_pos"]
K = 6  # small window around the position, no assumption about span size

def window(ids, pos, k=K):
    lo = max(0, pos - k)
    hi = min(len(ids), pos + k + 1)
    return lo, hi, ids[lo:hi]

lo_r, hi_r, win_r = window(ids_real, np_real)
lo_c, hi_c, win_c = window(ids_ctrl, np_ctrl)

print()
print(f"real  tokens around spec needle_pos={np_real}  (positions {lo_r}..{hi_r-1}):")
for i, tid in zip(range(lo_r, hi_r), win_r):
    marker = "  <-- needle_pos" if i == np_real else ""
    print(f"  [{i}] id={tid:>7d}  {tok.decode([tid])!r}{marker}")

print()
print(f"ctrl  tokens around spec needle_pos={np_ctrl}  (positions {lo_c}..{hi_c-1}):")
for i, tid in zip(range(lo_c, hi_c), win_c):
    marker = "  <-- needle_pos" if i == np_ctrl else ""
    print(f"  [{i}] id={tid:>7d}  {tok.decode([tid])!r}{marker}")

# --- if there is exactly one diff, show what and where -----------------------
if len(diff_positions) == 1:
    p = diff_positions[0]
    print()
    print(f"diff at position {p}:")
    print(f"  real id={ids_real[p]}  {tok.decode([ids_real[p]])!r}")

case: 'lantern' vs 'garden', pair readout lantern/torch, dist=64, filler=2

token ids: lantern=73165  garden=13551  torch=7834

spec['n_tokens']    : real=8292  ctrl=8292
spec['needle_pos']  : real=144  ctrl=144
tokenized length    : real=8292  ctrl=8292

positions differing between real and ctrl tokenized inputs: 1
differing positions: [144]

real  tokens around spec needle_pos=144  (positions 138..150):
  [138] id=  14401  ' wet'
  [139] id=     13  '.'
  [140] id=    576  ' The'
  [141] id=   3281  ' special'
  [142] id=   3409  ' word'
  [143] id=    374  ' is'
  [144] id=  73165  ' lantern'  <-- needle_pos
  [145] id=     13  '.'
  [146] id=    758  ' In'
  [147] id=    264  ' a'
  [148] id=  14178  ' hole'
  [149] id=    304  ' in'
  [150] id=    279  ' the'

ctrl  tokens around spec needle_pos=144  (positions 138..150):
  [138] id=  14401  ' wet'
  [139] id=     13  '.'
  [140] id=    576  ' The'
  [141] id=   3281  ' special'
  [142] id=   3409  ' word'
  [143] id=    374  ' is

In [47]:
print("compression_boundary:",
      spec_real["compression_boundary"],
      spec_ctrl["compression_boundary"])

print("actual_eviction_distance:",
      spec_real["actual_eviction_distance"],
      spec_ctrl["actual_eviction_distance"])

print("needle_is_evicted:",
      spec_real["needle_is_evicted"],
      spec_ctrl["needle_is_evicted"])

compression_boundary: 228 228
actual_eviction_distance: 84 84
needle_is_evicted: True True


In [48]:
device = next(bundle.model.parameters()).device
ins_real = {k: v.to(device) for k, v in ins_real.items()}
ins_ctrl = {k: v.to(device) for k, v in ins_ctrl.items()}

on_real = probe.run(
    ins_real,
    nowrite=False,
    layers=EXP["layers"],
    capture_residual=True
)

on_ctrl = probe.run(
    ins_ctrl,
    nowrite=False,
    layers=EXP["layers"],
    capture_residual=True
)

In [49]:
import torch.nn.functional as F

for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1).float()
    o_ctrl = on_ctrl.o_t(L, pos=-1).float()
    diff = o_real - o_ctrl

    print(f"\nLayer {L}")
    print("||o_t_real||:", o_real.norm().item())
    print("||o_t_ctrl||:", o_ctrl.norm().item())
    print("||diff||:", diff.norm().item())
    print("relative_diff:", diff.norm().item() / max(o_real.norm().item(), 1e-12))
    print("cosine:", F.cosine_similarity(o_real.flatten(), o_ctrl.flatten(), dim=0).item())


Layer 9
||o_t_real||: 0.6939501166343689
||o_t_ctrl||: 0.6869507431983948
||diff||: 0.0173657163977623
relative_diff: 0.02502444481454279
cosine: 0.9997351169586182

Layer 18
||o_t_real||: 1.1806039810180664
||o_t_ctrl||: 1.1499735116958618
||diff||: 0.1390652060508728
relative_diff: 0.11779157811322401
cosine: 0.9932234287261963

Layer 27
||o_t_real||: 3.6372334957122803
||o_t_ctrl||: 3.471142530441284
||diff||: 0.3218340575695038
relative_diff: 0.08848319964855018
cosine: 0.9969905614852905


In [50]:
needle_id = tokenizer.encode(
    f" {DEBUG_ORIGINAL}",
    add_special_tokens=False
)[0]

distractor_id = tokenizer.encode(
    f" {DEBUG_DISTRACTOR}",
    add_special_tokens=False
)[0]

print(
    "readout pair:",
    DEBUG_ORIGINAL, needle_id,
    "/",
    DEBUG_DISTRACTOR, distractor_id
)

for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1)
    o_ctrl = on_ctrl.o_t(L, pos=-1)

    lg_real = ai.readout_logits(
        o_real,
        bundle,
        lens=lens if EXP["use_jlens"] else None,
        layer=L
    )

    lg_ctrl = ai.readout_logits(
        o_ctrl,
        bundle,
        lens=lens if EXP["use_jlens"] else None,
        layer=L
    )

    p_real = torch.softmax(lg_real.float(), dim=-1)
    p_ctrl = torch.softmax(lg_ctrl.float(), dim=-1)

    real_log_ratio = (
        torch.log(p_real[needle_id])
        - torch.log(p_real[distractor_id])
    )

    ctrl_log_ratio = (
        torch.log(p_ctrl[needle_id])
        - torch.log(p_ctrl[distractor_id])
    )

    delta = real_log_ratio - ctrl_log_ratio

    print(f"\nLayer {L}")
    print(f"real p({DEBUG_ORIGINAL}):",
          p_real[needle_id].item())
    print(f"real p({DEBUG_DISTRACTOR}):",
          p_real[distractor_id].item())
    print(f"ctrl p({DEBUG_ORIGINAL}):",
          p_ctrl[needle_id].item())
    print(f"ctrl p({DEBUG_DISTRACTOR}):",
          p_ctrl[distractor_id].item())

    print("real log-ratio:", real_log_ratio.item())
    print("ctrl log-ratio:", ctrl_log_ratio.item())
    print("delta:", delta.item())

    print(
        f"real rank {DEBUG_ORIGINAL}:",
        ai.token_rank(lg_real, needle_id)
    )
    print(
        f"ctrl rank {DEBUG_ORIGINAL}:",
        ai.token_rank(lg_ctrl, needle_id)
    )

readout pair: lantern 73165 / torch 7834

Layer 9
real p(lantern): 2.079647498707738e-23
real p(torch): 3.4771405931621965e-22
ctrl p(lantern): 3.017800330241826e-23
ctrl p(torch): 4.874590899801156e-22
real log-ratio: -2.8165969848632812
ctrl log-ratio: -2.782093048095703
delta: -0.034503936767578125
real rank lantern: 116302
ctrl rank lantern: 115648

Layer 18
real p(lantern): 3.44575482813525e-07
real p(torch): 2.1844401842940897e-08
ctrl p(lantern): 5.697417293504259e-08
ctrl p(torch): 1.2194754095418148e-09
real log-ratio: 2.7583675384521484
ctrl log-ratio: 3.84417724609375
delta: -1.0858097076416016
real rank lantern: 17157
ctrl rank lantern: 28084

Layer 27
real p(lantern): 2.3229898005183713e-08
real p(torch): 1.322519693758295e-08
ctrl p(lantern): 4.11140135270216e-08
ctrl p(torch): 1.578422192949347e-08
real log-ratio: 0.5633163452148438
ctrl log-ratio: 0.9573383331298828
delta: -0.39402198791503906
real rank lantern: 96810
ctrl rank lantern: 86519


In [51]:
check_lantern = paired_content[
    (paired_content["original_needle"] == DEBUG_ORIGINAL) &
    (paired_content["requested_distance"] == DEBUG_DIST) &
    (paired_content["filler_idx"] == DEBUG_FILLER)
][[
    "layer",
    "original_log_ratio",
    "control_log_ratio",
    "delta_log_original_vs_control",
    "fold_original_vs_control",
    "original_eviction_distance",
    "control_eviction_distance",
]]

print(check_lantern.to_string(index=False))

 layer  original_log_ratio  control_log_ratio  delta_log_original_vs_control  fold_original_vs_control  original_eviction_distance  control_eviction_distance
     9           -2.816597          -2.782093                      -0.034504                  0.966085                          84                         84
    18            2.758368           3.844177                      -1.085809                  0.337629                          84                         84
    27            0.563316           0.957338                      -0.394022                  0.674339                          84                         84


### C2 five-link trace — negative lantern case

For `lantern` vs unrelated control `garden`
(requested distance 64, filler 2; readout pair `lantern/torch`):

1. **Input: PASS**
   - same sequence length
   - same needle position
   - exactly one token differs (`lantern` vs `garden`)

2. **Eviction: PASS**
   - same compression boundary
   - same actual eviction distance = 84
   - both stored words are evicted

3. **AHN output: DIFFERENCE PRESENT**
   - Layer 18 relative `o_t` difference ≈ 11.7%
   - cosine ≈ 0.9933
   - therefore the two conditions are distinguishable in AHN state space

4. **J-Lens readout: UNEXPECTED RELATIVE DIRECTION**
   - Layer 18 real log-ratio (`lantern/torch`) = 2.750920
   - control log-ratio = 3.800121
   - delta = -1.049201
   - the control state therefore shows a stronger relative `lantern/torch`
     preference than the actual lantern-stored state

5. **Control metric: PASS**
   - the live recomputation exactly matches `paired_content`
   - therefore the negative C2 value is not caused by summary/bookkeeping error

For this diagnostic case, the pipeline is structurally consistent through the
final metric. The unexpected C2 behavior appears downstream of a genuine AHN
state difference, in the relationship between that state difference and the
J-Lens target-vs-distractor readout.

In [52]:
needle_id = tokenizer.encode(
    f" {DEBUG_ORIGINAL}",
    add_special_tokens=False
)[0]

distractor_id = tokenizer.encode(
    f" {DEBUG_DISTRACTOR}",
    add_special_tokens=False
)[0]

print("pair:", DEBUG_ORIGINAL, "/", DEBUG_DISTRACTOR)

for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1)
    o_ctrl = on_ctrl.o_t(L, pos=-1)

    # J-Lens
    j_real = ai.readout_logits(
        o_real,
        bundle,
        lens=lens,
        layer=L
    )

    j_ctrl = ai.readout_logits(
        o_ctrl,
        bundle,
        lens=lens,
        layer=L
    )

    # Plain readout
    plain_real = ai.readout_logits(
        o_real,
        bundle,
        lens=None
    )

    plain_ctrl = ai.readout_logits(
        o_ctrl,
        bundle,
        lens=None
    )

    j_delta = (
        (j_real[needle_id] - j_real[distractor_id])
        -
        (j_ctrl[needle_id] - j_ctrl[distractor_id])
    )

    plain_delta = (
        (plain_real[needle_id] - plain_real[distractor_id])
        -
        (plain_ctrl[needle_id] - plain_ctrl[distractor_id])
    )

    print(f"\nLayer {L}")
    print("J-Lens delta:", j_delta.item())
    print("Plain delta: ", plain_delta.item())

    print("J real rank needle:",
          ai.token_rank(j_real, needle_id))
    print("J ctrl rank needle:",
          ai.token_rank(j_ctrl, needle_id))

    print("Plain real rank needle:",
          ai.token_rank(plain_real, needle_id))
    print("Plain ctrl rank needle:",
          ai.token_rank(plain_ctrl, needle_id))

pair: lantern / torch

Layer 9
J-Lens delta: -0.034500837326049805
Plain delta:  -0.021044492721557617
J real rank needle: 116302
J ctrl rank needle: 115648
Plain real rank needle: 41815
Plain ctrl rank needle: 41800

Layer 18
J-Lens delta: -1.0858073234558105
Plain delta:  -0.5989856719970703
J real rank needle: 17157
J ctrl rank needle: 28084
Plain real rank needle: 2814
Plain ctrl rank needle: 2989

Layer 27
J-Lens delta: -0.3940219283103943
Plain delta:  -0.4905972480773926
J real rank needle: 96810
J ctrl rank needle: 86519
Plain real rank needle: 101758
Plain ctrl rank needle: 94456


### Plain-readout vs J-Lens diagnostic

For the strongest negative C2 case (`lantern` vs `garden`,
readout pair `lantern/torch`), the negative corrected effect is present
under both readout methods.

Layer 18:
- J-Lens delta = -1.0492
- Plain readout delta = -0.6560

Therefore, the negative C2 direction is not created solely by the J-Lens
transformation. The same direction is already present when the AHN `o_t`
state is decoded with the plain vocabulary readout.

This shifts the likely explanation away from a J-Lens-specific sign reversal
and toward either content-dependent AHN state geometry or the target-vs-
distractor control metric itself.

In [53]:
L = 18

o_real = on_real.o_t(L, pos=-1)
o_ctrl = on_ctrl.o_t(L, pos=-1)

lg_real = ai.readout_logits(
    o_real,
    bundle,
    lens=lens,
    layer=L
)

lg_ctrl = ai.readout_logits(
    o_ctrl,
    bundle,
    lens=lens,
    layer=L
)

lantern_real = lg_real[needle_id].item()
lantern_ctrl = lg_ctrl[needle_id].item()

torch_real = lg_real[distractor_id].item()
torch_ctrl = lg_ctrl[distractor_id].item()

print("Layer 18")
print("lantern real logit:", lantern_real)
print("lantern ctrl logit:", lantern_ctrl)
print("lantern change real-ctrl:", lantern_real - lantern_ctrl)

print()
print("torch real logit:", torch_real)
print("torch ctrl logit:", torch_ctrl)
print("torch change real-ctrl:", torch_real - torch_ctrl)

print()
print("pair delta:",
      (lantern_real - torch_real) -
      (lantern_ctrl - torch_ctrl))

Layer 18
lantern real logit: 5.024381637573242
lantern ctrl logit: 4.226924419403076
lantern change real-ctrl: 0.797457218170166

torch real logit: 2.2660131454467773
torch ctrl logit: 0.382748544216156
torch change real-ctrl: 1.8832646012306213

pair delta: -1.0858073830604553


### Why the Layer-18 C2 delta is negative

For the `lantern` vs `garden` condition:

- `lantern` logit increases by +0.687
- `torch` logit increases by +1.736

Therefore the stored `lantern` does strengthen the target token, but it
strengthens the semantically related distractor `torch` even more.

This produces the negative corrected pair delta:

`(+0.687) - (+1.736) = -1.049`

So this C2 failure is not simply "the model does not retain lantern."
Instead, the target-vs-distractor metric is strongly affected by how the
stored content changes both members of the semantic pair.

In [54]:
pairs = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

check = (
    paired_content[
        paired_content["layer"] == 18
    ]
    .groupby("original_needle")
    .agg(
        mean_original_log_ratio=("original_log_ratio", "mean"),
        mean_control_log_ratio=("control_log_ratio", "mean"),
        mean_delta=("delta_log_original_vs_control", "mean"),
        min_delta=("delta_log_original_vs_control", "min"),
        max_delta=("delta_log_original_vs_control", "max"),
        n=("delta_log_original_vs_control", "size"),
    )
    .reset_index()
)

print(check.to_string(index=False))

original_needle  mean_original_log_ratio  mean_control_log_ratio  mean_delta  min_delta  max_delta  n
          Paris                -0.584943               -0.633335    0.048392  -0.449238   0.384117 21
          Tokyo                -1.274131               -1.258824   -0.015307  -0.297148   0.310827 21
         banana                -0.654945               -0.700912    0.045967  -0.113280   0.447905 21
        lantern                 0.849178                0.888072   -0.038894  -1.085809   0.433518 21


In [55]:
best = (
    paired_content[
        paired_content["layer"] == 18
    ]
    .sort_values("delta_log_original_vs_control", ascending=False)
    .head(5)
)

print(
    best[
        [
            "original_needle",
            "requested_distance",
            "filler_idx",
            "original_log_ratio",
            "control_log_ratio",
            "delta_log_original_vs_control",
        ]
    ].to_string(index=False)
)

original_needle  requested_distance  filler_idx  original_log_ratio  control_log_ratio  delta_log_original_vs_control
         banana                  64           2           -0.407268          -0.855173                       0.447905
        lantern                 256           0            2.774419           2.340900                       0.433518
          Paris                  64           0            0.997193           0.613076                       0.384117
        lantern                 512           0            1.675482           1.342409                       0.333073
          Tokyo                  64           2            0.109768          -0.201059                       0.310827


In [56]:
# ============================================================
# C2 positive comparison case — Link 1: input equivalence
# ============================================================

DEBUG_ORIGINAL   = "lantern"
DEBUG_CONTROL    = "garden"
DEBUG_DISTRACTOR = "torch"
DEBUG_DIST       = 256
DEBUG_FILLER     = 0

# Build prompts
spec_real = ai.build_niah_prompt(
    tokenizer,
    DEBUG_ORIGINAL,
    bundle,
    eviction_distance=DEBUG_DIST,
    in_window=False,
    filler_idx=DEBUG_FILLER,
)

spec_ctrl = ai.build_niah_prompt(
    tokenizer,
    DEBUG_CONTROL,
    bundle,
    eviction_distance=DEBUG_DIST,
    in_window=False,
    filler_idx=DEBUG_FILLER,
)

# Tokenize
ins_real = tokenizer(
    spec_real["prompt"],
    return_tensors="pt",
)

ins_ctrl = tokenizer(
    spec_ctrl["prompt"],
    return_tensors="pt",
)

# Token IDs for the C2 pair
original_id = tokenizer.encode(
    f" {DEBUG_ORIGINAL}",
    add_special_tokens=False
)[0]

control_id = tokenizer.encode(
    f" {DEBUG_CONTROL}",
    add_special_tokens=False
)[0]

distractor_id = tokenizer.encode(
    f" {DEBUG_DISTRACTOR}",
    add_special_tokens=False
)[0]

print(
    f"case: '{DEBUG_ORIGINAL}' vs '{DEBUG_CONTROL}', "
    f"pair readout {DEBUG_ORIGINAL}/{DEBUG_DISTRACTOR}, "
    f"dist={DEBUG_DIST}, filler={DEBUG_FILLER}"
)

print(
    f"\ntoken ids: "
    f"{DEBUG_ORIGINAL}={original_id}  "
    f"{DEBUG_CONTROL}={control_id}  "
    f"{DEBUG_DISTRACTOR}={distractor_id}"
)

print("\nspec['n_tokens']    :",
      f"real={spec_real['n_tokens']}  ctrl={spec_ctrl['n_tokens']}")

print("spec['needle_pos']  :",
      f"real={spec_real['needle_pos']}  ctrl={spec_ctrl['needle_pos']}")

print("tokenized length    :",
      f"real={ins_real['input_ids'].shape[1]}  "
      f"ctrl={ins_ctrl['input_ids'].shape[1]}")

# Find every position that differs
ids_real = ins_real["input_ids"][0]
ids_ctrl = ins_ctrl["input_ids"][0]

diff_positions = (
    ids_real != ids_ctrl
).nonzero(as_tuple=True)[0].tolist()

print(
    "\npositions differing between real and ctrl tokenized inputs:",
    len(diff_positions)
)
print("differing positions:", diff_positions)

# Inspect around the expected needle location
for name, spec, ids in [
    ("real", spec_real, ids_real),
    ("ctrl", spec_ctrl, ids_ctrl),
]:
    pos = spec["needle_pos"]
    start = max(0, pos - 6)
    end = min(len(ids), pos + 7)

    print(
        f"\n{name} tokens around spec needle_pos={pos} "
        f"(positions {start}..{end-1}):"
    )

    for i in range(start, end):
        token_text = tokenizer.decode([int(ids[i])])
        marker = "  <-- needle_pos" if i == pos else ""
        print(
            f"  [{i}] id={int(ids[i]):7d}  "
            f"{token_text!r}{marker}"
        )

if len(diff_positions) == 1:
    p = diff_positions[0]
    print(f"\ndiff at position {p}:")
    print(
        f"  real id={int(ids_real[p])}  "
        f"{tokenizer.decode([int(ids_real[p])])!r}"
    )
    print(
        f"  ctrl id={int(ids_ctrl[p])}  "
        f"{tokenizer.decode([int(ids_ctrl[p])])!r}"
    )

case: 'lantern' vs 'garden', pair readout lantern/torch, dist=256, filler=0

token ids: lantern=73165  garden=13551  torch=7834

spec['n_tokens']    : real=8484  ctrl=8484
spec['needle_pos']  : real=148  ctrl=148
tokenized length    : real=8484  ctrl=8484

positions differing between real and ctrl tokenized inputs: 1
differing positions: [148]

real tokens around spec needle_pos=148 (positions 142..154):
  [142] id=   1549  ' again'
  [143] id=     13  '.'
  [144] id=    576  ' The'
  [145] id=   3281  ' special'
  [146] id=   3409  ' word'
  [147] id=    374  ' is'
  [148] id=  73165  ' lantern'  <-- needle_pos
  [149] id=     13  '.'
  [150] id=    576  ' The'
  [151] id=  16359  ' grass'
  [152] id=    374  ' is'
  [153] id=   6176  ' green'
  [154] id=     13  '.'

ctrl tokens around spec needle_pos=148 (positions 142..154):
  [142] id=   1549  ' again'
  [143] id=     13  '.'
  [144] id=    576  ' The'
  [145] id=   3281  ' special'
  [146] id=   3409  ' word'
  [147] id=    374  

In [57]:
print("compression_boundary:",
      spec_real["compression_boundary"],
      spec_ctrl["compression_boundary"])

print("actual_eviction_distance:",
      spec_real["actual_eviction_distance"],
      spec_ctrl["actual_eviction_distance"])

print("needle_is_evicted:",
      spec_real["needle_is_evicted"],
      spec_ctrl["needle_is_evicted"])

compression_boundary: 420 420
actual_eviction_distance: 272 272
needle_is_evicted: True True


In [58]:
device = next(bundle.model.parameters()).device
ins_real = {k: v.to(device) for k, v in ins_real.items()}
ins_ctrl = {k: v.to(device) for k, v in ins_ctrl.items()}

on_real = probe.run(
    ins_real,
    nowrite=False,
    layers=EXP["layers"],
    capture_residual=True
)

on_ctrl = probe.run(
    ins_ctrl,
    nowrite=False,
    layers=EXP["layers"],
    capture_residual=True
)

In [59]:
import torch.nn.functional as F

for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1).float()
    o_ctrl = on_ctrl.o_t(L, pos=-1).float()

    diff = o_real - o_ctrl

    print(f"\nLayer {L}")
    print("||o_t_real||:", o_real.norm().item())
    print("||o_t_ctrl||:", o_ctrl.norm().item())
    print("||diff||:", diff.norm().item())
    print(
        "relative_diff:",
        diff.norm().item() / max(o_real.norm().item(), 1e-12)
    )
    print(
        "cosine:",
        F.cosine_similarity(
            o_real.flatten(),
            o_ctrl.flatten(),
            dim=0
        ).item()
    )


Layer 9
||o_t_real||: 0.5092387795448303
||o_t_ctrl||: 0.5074635744094849
||diff||: 0.019234871491789818
relative_diff: 0.03777181209369483
cosine: 0.9992902278900146

Layer 18
||o_t_real||: 0.9615176320075989
||o_t_ctrl||: 0.9120883941650391
||diff||: 0.12746429443359375
relative_diff: 0.13256573794435253
cosine: 0.992129921913147

Layer 27
||o_t_real||: 2.5726609230041504
||o_t_ctrl||: 2.7333054542541504
||diff||: 0.2883349657058716
relative_diff: 0.1120765519962797
cosine: 0.9959235787391663


In [60]:
L = 18

needle_id = tokenizer.encode(
    f" {DEBUG_ORIGINAL}",
    add_special_tokens=False
)[0]

distractor_id = tokenizer.encode(
    f" {DEBUG_DISTRACTOR}",
    add_special_tokens=False
)[0]

o_real = on_real.o_t(L, pos=-1)
o_ctrl = on_ctrl.o_t(L, pos=-1)

lg_real = ai.readout_logits(
    o_real,
    bundle,
    lens=lens,
    layer=L
)

lg_ctrl = ai.readout_logits(
    o_ctrl,
    bundle,
    lens=lens,
    layer=L
)

needle_real = lg_real[needle_id].item()
needle_ctrl = lg_ctrl[needle_id].item()

dist_real = lg_real[distractor_id].item()
dist_ctrl = lg_ctrl[distractor_id].item()

needle_change = needle_real - needle_ctrl
dist_change = dist_real - dist_ctrl
delta = needle_change - dist_change

print("Layer 18")
print("target:", DEBUG_ORIGINAL)
print("distractor:", DEBUG_DISTRACTOR)

print("\ntarget real logit:", needle_real)
print("target ctrl logit:", needle_ctrl)
print("target change:", needle_change)

print("\ndistractor real logit:", dist_real)
print("distractor ctrl logit:", dist_ctrl)
print("distractor change:", dist_change)

print("\npair delta:", delta)
print("real target rank:", ai.token_rank(lg_real, needle_id))
print("ctrl target rank:", ai.token_rank(lg_ctrl, needle_id))

Layer 18
target: lantern
distractor: torch

target real logit: -1.9364755153656006
target ctrl logit: -1.7101178169250488
target change: -0.22635769844055176

distractor real logit: -4.710894584655762
distractor ctrl logit: -4.051018238067627
distractor change: -0.6598763465881348

pair delta: 0.433518648147583
real target rank: 111008
ctrl target rank: 109066


### C2 debugging conclusion

The C2 pipeline was traced through input construction, eviction, AHN output,
J-Lens readout, and the final saved metric.

For both a strongly negative and a strongly positive `lantern/torch` condition:

- input construction is structurally matched
- eviction behavior is matched
- `o_t` differs between the real and content-control conditions
- the J-Lens readout reproduces the saved C2 metric
- the saved metric bookkeeping is correct

The difference between positive and negative C2 conditions comes from how the
target and distractor logits move relative to each other.

Strong negative condition:
- lantern logit change = +0.687
- torch logit change = +1.736
- pair delta = -1.049

Strong positive condition:
- lantern logit change = -0.189
- torch logit change = -0.660
- pair delta = +0.471

Therefore, the sign of the C2 statistic does not simply indicate whether the
stored needle became stronger in the AHN state. It depends on the relative
movement of the target and its semantic distractor.

For the cases inspected, no implementation or bookkeeping bug was found in the
five-link pipeline. The main issue appears to be that the target-vs-distractor
control metric is highly content- and condition-dependent.

This does not establish that J-Lens is valid or that AHN retains no information.
It shows that the current C2 statistic is not a clean memory-specific readout.

In [61]:
print(df_content_control.columns.tolist())

['original_needle', 'control_needle', 'tested_needle', 'distractor', 'layer', 'requested_distance', 'actual_eviction_distance', 'filler_idx', 'n_tokens', 'needle_pos', 'p_needle', 'p_distractor', 'log_ratio']


In [62]:
print(df_bias.columns.tolist())

['stored_needle', 'tested_needle', 'distractor', 'layer', 'requested_distance', 'actual_eviction_distance', 'filler_idx', 'p_needle', 'p_distractor', 'ratio']


In [63]:
import numpy as np

# Original condition:
# actual target is stored AND that same target is tested
orig = df_bias[
    df_bias["stored_needle"] == df_bias["tested_needle"]
][[
    "stored_needle",
    "layer",
    "requested_distance",
    "filler_idx",
    "p_needle",
]].copy()

orig = orig.rename(columns={
    "stored_needle": "original_needle",
    "p_needle": "p_target_original",
})

# Control condition:
# unrelated control word is stored, but original target is still tested
ctrl = df_content_control[
    df_content_control["original_needle"] == df_content_control["tested_needle"]
][[
    "original_needle",
    "layer",
    "requested_distance",
    "filler_idx",
    "p_needle",
]].copy()

ctrl = ctrl.rename(columns={
    "p_needle": "p_target_control",
})

# Match exact conditions
target_only = orig.merge(
    ctrl,
    on=[
        "original_needle",
        "layer",
        "requested_distance",
        "filler_idx",
    ],
    validate="one_to_one",
)

# Positive = target is stronger when target itself was stored
target_only["delta_log_target"] = (
    np.log(target_only["p_target_original"] + 1e-30)
    - np.log(target_only["p_target_control"] + 1e-30)
)

summary_target = (
    target_only
    .groupby(["original_needle", "layer"])["delta_log_target"]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
)

print(summary_target.to_string(index=False))

original_needle  layer      mean      std       min      max  count
          Paris      9  0.027612 0.287253 -0.549156 0.721222     21
          Paris     18  0.091672 0.441378 -0.748538 0.964399     21
          Paris     27 -0.097079 0.430874 -0.948310 0.613935     21
          Tokyo      9 -0.450349 0.715928 -2.957610 0.263734     21
          Tokyo     18 -0.308084 0.506546 -1.448611 0.676302     21
          Tokyo     27 -0.170957 0.353642 -1.023401 0.502520     21
         banana      9  0.002058 0.249280 -0.289175 0.845430     21
         banana     18  0.135432 0.682667 -1.244568 1.487549     21
         banana     27 -0.045603 0.445836 -1.148140 0.943466     21
        lantern      9 -0.036297 0.217519 -0.549770 0.467644     21
        lantern     18  0.016664 0.649439 -1.108855 1.799715     21
        lantern     27 -0.205191 0.353801 -1.218793 0.223957     21


In [64]:
import numpy as np

# Keep Layer 18 first because that is where the corrected C2 effect appeared.
L = 18

rows = []

for target in ["Paris", "Tokyo", "banana", "lantern"]:

    # Target readout when target itself was stored
    actual = df_bias[
        (df_bias["layer"] == L) &
        (df_bias["tested_needle"] == target) &
        (df_bias["stored_needle"] == target)
    ][[
        "requested_distance",
        "filler_idx",
        "p_needle"
    ]].rename(columns={
        "p_needle": "p_target_actual"
    })

    # Same target readout when some OTHER needle was stored
    others = df_bias[
        (df_bias["layer"] == L) &
        (df_bias["tested_needle"] == target) &
        (df_bias["stored_needle"] != target)
    ][[
        "stored_needle",
        "requested_distance",
        "filler_idx",
        "p_needle"
    ]].rename(columns={
        "stored_needle": "control_stored",
        "p_needle": "p_target_other"
    })

    merged = others.merge(
        actual,
        on=["requested_distance", "filler_idx"],
        validate="many_to_one"
    )

    merged["target"] = target

    merged["delta_log_target"] = (
        np.log(merged["p_target_actual"] + 1e-30)
        -
        np.log(merged["p_target_other"] + 1e-30)
    )

    rows.append(merged)

cross_control = pd.concat(rows, ignore_index=True)

summary = (
    cross_control
    .groupby(["target", "control_stored"])["delta_log_target"]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
)

print(summary.to_string(index=False))

 target control_stored      mean      std       min      max  count
  Paris          Tokyo  0.173129 0.391862 -0.561095 1.394375     21
  Paris         banana -0.013775 0.541103 -1.249296 1.756460     21
  Paris        lantern  0.039931 0.566025 -1.183323 1.685468     21
  Tokyo          Paris  0.045124 0.384812 -1.059861 1.071589     21
  Tokyo         banana -0.259444 0.435244 -1.214846 0.483167     21
  Tokyo        lantern  0.015081 0.474615 -0.966405 0.897923     21
 banana          Paris  0.289613 0.698037 -1.476629 1.534132     21
 banana          Tokyo  0.272939 0.446867 -0.465274 1.499535     21
 banana        lantern  0.123432 0.437305 -0.745679 1.056282     21
lantern          Paris  0.143499 0.508461 -1.077041 1.636242     21
lantern          Tokyo  0.115798 0.379103 -0.604606 0.961631     21
lantern         banana -0.052176 0.311205 -0.717181 0.354008     21


In [65]:
CONTROL_CANDIDATES = [
    "river",
    "chair",
    "window",
    "garden",
    "table",
    "house",
    "water",
    "paper",
    "stone",
    "music",
    "green",
    "cloud",
]

targets = ["Paris", "Tokyo", "banana", "lantern"]

valid_controls = []

for word in CONTROL_CANDIDATES:
    ids = tokenizer.encode(
        f" {word}",
        add_special_tokens=False
    )

    # must be exactly one token
    if len(ids) != 1:
        continue

    # don't allow target words themselves
    if word in targets:
        continue

    valid_controls.append((word, ids[0]))

print("Valid single-token controls:")
for word, tid in valid_controls:
    print(f"{word:10s} -> {tid}")

Valid single-token controls:
river      -> 14796
chair      -> 10496
window     -> 3241
garden     -> 13551
table      -> 1965
house      -> 3753
water      -> 3015
paper      -> 5567
stone      -> 9798
music      -> 4627
green      -> 6176
cloud      -> 9437


In [66]:
PILOT_CONTROLS = ["river", "chair", "window", "garden"]
TARGETS = ["Paris", "Tokyo", "banana", "lantern"]

problems = []

for target in TARGETS:
    for control in PILOT_CONTROLS:
        for distance in EXP["eviction_distances"]:
            for filler_idx in range(EXP["n_filler_variants"]):

                spec = ai.build_niah_prompt(
                    tokenizer,
                    control,
                    bundle,
                    eviction_distance=distance,
                    in_window=False,
                    filler_idx=filler_idx,
                )

                prompt_lower = spec["prompt"].lower()

                # target should not accidentally already appear
                if target.lower() in prompt_lower:
                    problems.append(
                        (target, control, distance, filler_idx, "target leaked")
                    )

                # control should appear exactly once as the stored word
                if prompt_lower.count(control.lower()) != 1:
                    problems.append(
                        (
                            target,
                            control,
                            distance,
                            filler_idx,
                            f"control count={prompt_lower.count(control.lower())}",
                        )
                    )

print("Problems found:", len(problems))

for row in problems[:20]:
    print(row)

if not problems:
    print("PASS: pilot control prompts are structurally clean.")

Problems found: 0
PASS: pilot control prompts are structurally clean.


In [67]:
# ============================================================
# C2-v2 PILOT — multi-control target-only baseline
# 84 forward passes total
# ============================================================

import pandas as pd

PILOT_CONTROLS = ["river", "chair", "window", "garden"]
TARGETS = ["Paris", "Tokyo", "banana", "lantern"]
LAYERS = [9, 18, 27]

target_ids = {
    target: tokenizer.encode(
        f" {target}",
        add_special_tokens=False
    )[0]
    for target in TARGETS
}

pilot_rows = []

total = (
    len(PILOT_CONTROLS)
    * len(EXP["eviction_distances"])
    * EXP["n_filler_variants"]
)

done = 0

for control in PILOT_CONTROLS:

    for distance in EXP["eviction_distances"]:

        for filler_idx in range(EXP["n_filler_variants"]):

            spec = ai.build_niah_prompt(
                tokenizer,
                control,
                bundle,
                eviction_distance=distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            assert spec["needle_is_evicted"], (
                control,
                distance,
                filler_idx,
                spec["actual_eviction_distance"],
            )

            ins = tokenizer(
                spec["prompt"],
                return_tensors="pt",
            ).to(bundle.model.device)

            on = probe.run(
                ins,
                nowrite=False,
                layers=LAYERS,
                capture_residual=False,
            )

            # One AHN state -> score ALL four targets
            for L in LAYERS:

                o_t = on.o_t(L, pos=-1)

                logits = ai.readout_logits(
                    o_t,
                    bundle,
                    lens=lens if EXP["use_jlens"] else None,
                    layer=L,
                )

                for target in TARGETS:

                    p_target = ai.token_prob(
                        logits,
                        target_ids[target],
                    )

                    pilot_rows.append({
                        "control_needle": control,
                        "tested_target": target,
                        "layer": L,
                        "requested_distance": distance,
                        "actual_eviction_distance":
                            spec["actual_eviction_distance"],
                        "filler_idx": filler_idx,
                        "p_target": p_target,
                    })

            done += 1

            print(
                f"\r{done}/{total} forward passes",
                end="",
                flush=True,
            )

print("\nDone.")

df_c2_multicontrol = pd.DataFrame(pilot_rows)

print("rows:", len(df_c2_multicontrol))
print(df_c2_multicontrol.head())

84/84 forward passes
Done.
rows: 1008
  control_needle tested_target  layer  requested_distance  \
0          river         Paris      9                  64   
1          river         Tokyo      9                  64   
2          river        banana      9                  64   
3          river       lantern      9                  64   
4          river         Paris     18                  64   

   actual_eviction_distance  filler_idx      p_target  
0                        80           0  9.887024e-22  
1                        80           0  1.127859e-21  
2                        80           0  2.198482e-22  
3                        80           0  5.710460e-22  
4                        80           0  1.508534e-08  


In [68]:
# ------------------------------------------------------------
# Original condition: target itself was stored
# ------------------------------------------------------------
orig = df_bias[
    df_bias["stored_needle"] == df_bias["tested_needle"]
][[
    "stored_needle",
    "layer",
    "requested_distance",
    "filler_idx",
    "p_needle",
]].copy()

orig = orig.rename(columns={
    "stored_needle": "target",
    "p_needle": "p_target_original",
})

# ------------------------------------------------------------
# Multi-control baseline:
# mean log p(target) across the 4 unrelated controls
# ------------------------------------------------------------
ctrl = df_c2_multicontrol.copy()

ctrl["log_p_target"] = np.log(ctrl["p_target"] + 1e-30)

ctrl_mean = (
    ctrl
    .groupby([
        "tested_target",
        "layer",
        "requested_distance",
        "filler_idx",
    ])
    .agg(
        mean_log_p_control=("log_p_target", "mean"),
        std_log_p_control=("log_p_target", "std"),
        n_controls=("control_needle", "nunique"),
    )
    .reset_index()
    .rename(columns={"tested_target": "target"})
)

# ------------------------------------------------------------
# Match original condition to multi-control baseline
# ------------------------------------------------------------
c2_v2 = orig.merge(
    ctrl_mean,
    on=[
        "target",
        "layer",
        "requested_distance",
        "filler_idx",
    ],
    validate="one_to_one",
)

c2_v2["delta_log_target_multicontrol"] = (
    np.log(c2_v2["p_target_original"] + 1e-30)
    - c2_v2["mean_log_p_control"]
)

# ------------------------------------------------------------
# Summary by target/layer
# ------------------------------------------------------------
summary_c2_v2 = (
    c2_v2
    .groupby(["target", "layer"])["delta_log_target_multicontrol"]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
)

print(summary_c2_v2.to_string(index=False))

 target  layer      mean      std       min      max  count
  Paris      9 -0.102182 0.250971 -0.598789 0.290189     21
  Paris     18  0.080056 0.300457 -0.544656 0.537566     21
  Paris     27 -0.220255 0.438089 -1.112548 0.283193     21
  Tokyo      9 -0.191489 0.402647 -1.298378 0.406292     21
  Tokyo     18 -0.036374 0.413458 -0.907948 0.883949     21
  Tokyo     27 -0.217524 0.329060 -1.106196 0.311374     21
 banana      9  0.018791 0.218393 -0.410500 0.614152     21
 banana     18  0.102842 0.543388 -1.267545 1.317269     21
 banana     27 -0.028124 0.320601 -0.747376 0.703075     21
lantern      9 -0.106732 0.167334 -0.459146 0.226806     21
lantern     18  0.031042 0.459963 -0.983784 1.359018     21
lantern     27 -0.137408 0.244519 -0.896470 0.119645     21


## C2 Debugging Summary

We finished debugging **C2** through the full pipeline.

### What we checked

1. **Input**
   - Real and control prompts were structurally matched.
   - Only the stored word changed.

2. **Eviction**
   - Real and control conditions used the same compression boundary.
   - Both stored words were actually evicted at the same distance.

3. **AHN state (`o_t`)**
   - The AHN state changed between the real and control conditions.
   - So the model was not producing identical memory states.

4. **J-Lens readout**
   - J-Lens preserved differences between those AHN states.
   - The live J-Lens calculations matched the saved C2 results.

5. **Final metric**
   - The saved C2 calculations were correct.
   - We did not find a bookkeeping or aggregation bug.

### Why C2 is unstable

For some conditions, storing the correct target increased the target signal, but it also increased the semantic distractor even more.

Example at Layer 18 for `lantern/torch`:

- lantern logit change: `+0.687`
- torch logit change: `+1.736`
- resulting C2 delta: `-1.049`

So a negative C2 score does not necessarily mean the target was forgotten.

We also checked the same AHN states using the **plain vocabulary readout** instead of J-Lens. The negative direction was still present, so this is **not a J-Lens-only problem**.

### Attempts to improve C2

We tested:

- target-only comparison
- using other stored needles as controls
- a multi-control baseline using 4 unrelated control words

For the multi-control pilot we ran:

- **84 additional model forward passes**
- **1008 readout rows**
- Layers 9, 18, and 27

At Layer 18, the average multi-control effects were:

- Paris: `+0.049`
- Tokyo: `-0.026`
- banana: `+0.147`
- lantern: `+0.031`

The results were still mixed and highly variable.

### Current conclusion

We did **not** find evidence that C2 is failing because of:

- an input-construction bug
- an eviction mismatch
- a bookkeeping bug
- a J-Lens-only sign reversal

Instead, the current C2 readout appears to be **content-dependent and not consistently memory-specific**.

Even replacing the single control with multiple unrelated controls did not produce a stable positive signal.

### Status

**C2 debugging is complete.**

The current C2 should be treated as an **insufficient/failed memory-specific control**, rather than continuing to modify it until it passes.

Next step: **debug C3 using the same step-by-step pipeline.**

In [69]:
import numpy as np
import pandas as pd
from scipy import stats

# ============================================================
# FINAL C2-v2 CONDITION-LEVEL ANALYSIS
# ============================================================

# 1) Average the 4 targets within each distance × filler condition
condition_level = (
    c2_v2
    .groupby([
        "layer",
        "requested_distance",
        "filler_idx",
    ])["delta_log_target_multicontrol"]
    .mean()
    .reset_index(name="condition_delta")
)

print("condition counts per layer:")
print(condition_level.groupby("layer").size())
print()

# 2) Summary + 95% CI + one-sample t-test against 0
final_rows = []

for L in sorted(condition_level["layer"].unique()):

    x = condition_level.loc[
        condition_level["layer"] == L,
        "condition_delta"
    ].to_numpy()

    n = len(x)
    mean_log = x.mean()
    sd = x.std(ddof=1)
    se = sd / np.sqrt(n)

    tcrit = stats.t.ppf(0.975, df=n - 1)

    ci_low_log = mean_log - tcrit * se
    ci_high_log = mean_log + tcrit * se

    t_stat, p_value = stats.ttest_1samp(x, popmean=0.0)

    final_rows.append({
        "layer": L,
        "n_conditions": n,
        "mean_log_effect": mean_log,
        "fold_change": np.exp(mean_log),
        "ci_low_fold": np.exp(ci_low_log),
        "ci_high_fold": np.exp(ci_high_log),
        "t_stat": t_stat,
        "p_value": p_value,
    })

final_c2_v2 = pd.DataFrame(final_rows)

print(final_c2_v2.to_string(
    index=False,
    formatters={
        "mean_log_effect": "{:.6f}".format,
        "fold_change": "{:.4f}".format,
        "ci_low_fold": "{:.4f}".format,
        "ci_high_fold": "{:.4f}".format,
        "t_stat": "{:.4f}".format,
        "p_value": "{:.6g}".format,
    }
))

condition counts per layer:
layer
9     21
18    21
27    21
dtype: int64

 layer  n_conditions mean_log_effect fold_change ci_low_fold ci_high_fold  t_stat  p_value
     9            21       -0.095403      0.9090      0.8433       0.9798 -2.6526 0.015279
    18            21        0.044391      1.0454      0.9017       1.2120  0.6262  0.53829
    27            21       -0.150828      0.8600      0.7596       0.9737 -2.5335 0.019764


In [ ]:
### C2-v2 Final Result

The multi-control target-only redesign was evaluated at the condition level
(21 distance × filler observations per layer).

- Layer 9: 0.909×, 95% CI [0.843, 0.980], p = 0.015
- Layer 18: 1.045×, 95% CI [0.902, 1.212], p = 0.538
- Layer 27: 0.860×, 95% CI [0.760, 0.974], p = 0.020

The expected positive retention-specific effect is therefore not reliably
supported. Layer 18 remains directionally positive but highly variable, while
layers 9 and 27 show negative effects.

Combined with the five-link debugging trace, this does not indicate an input,
eviction, bookkeeping, or J-Lens-only implementation failure. Rather, the C2
control does not produce the expected memory-specific signal under the current
readout/control design.

C2 debugging is stopped here; no further GPU reruns are justified for this
control.